# Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime
import itertools

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.colors as pc
import seaborn as sns

from scipy import stats
from scipy.integrate import trapezoid
from scipy.stats import gaussian_kde, mannwhitneyu, ks_2samp, pearsonr, spearmanr, kruskal
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, confusion_matrix

from utils import *

warnings.filterwarnings('ignore')

# Funções auxiliares

Todas as funções do estudo ficam em **`utils.py`**, importado na célula acima com `from utils import *`.

| Bloco | Funções |
|---|---|
| Filtro de Hampel | `hampel`, `ResultadoHampel` |
| Carga e limpeza | `carregar_crystallizer`, `aplicar_tratamentos`, `carregar_eventos` |
| Janelas e eventos | `valores_na_janela`, `adicionar_eventos_ultrapassagem`, `separar_eventos_por_data`, `unificar_eventos` |
| Inspeção (fonte dos eventos) | `carregar_inspecoes`, `identificar_lacunas`, `identificar_paradas_de_planta`, `montar_tabela_eventos`, `eventos_para_notebook` |
| Limpeza de excursões | `classificar_amostras_altas`, `limpar_excursoes` |
| Baseline e detectores | `avaliar_baseline_lift`, `criterios_padrao`, `avaliar_detector_diario`, `avaliar_detector_composto`, `avaliar_detector_adaptativo`, `comparar_detectores` |
| Gráficos da série | `plot_crystallizer`, `plot_crystallizers`, `plot_mm_crystallizer`, `plot_mm_crystallizers`, `plot_crystallizer_unificado` |
| Estatística | `estatisticas`, `plot_violin_classes`, `plot_violin_serie_vs_ultrapassagem`, `testar_classes_por_janela`, `separabilidade_features` |
| Não supervisionado | `gerar_janelas_deslizantes`, `janelas_nas_falhas`, `clusterizar_regimes`, `plotar_regimes_pca`, `calcular_margem_cross_reator`, `baseline_movel`, `avaliar_lift_series`, `campanhas_por_reator`, `perfil_hazard_campanha` |
| Classificação | `extrair_features`, `dividir_dados`, `selecionar_features`, `definir_modelos_e_grids`, `avaliar_cv_com_grid`, `avaliar_teste`, `rodar_classificacao` |
| Detecção de anomalia | `rodar_iforest_janelas`, `plotar_iforest_janelas` |
| Cartas de controle | `calcular_ewma`, `otimizar_ewma`, `plotar_carta_ewma`, `avaliar_carta_controle`, `calcular_cusum_dinamico`, `otimizar_cusum_dinamico`, `plotar_carta_cusum` |

# Carregando dados

## Política de tratamento de dados — onde cada versão pode ser usada

O sinal que antecipa falha é justamente a leitura alta de ferro: **o outlier é o alvo**, e filtro
estatístico de outlier e detector de falha são a mesma operação com sinais trocados. Verificação
feita contra as duas leituras que a planilha de inspeção confirma como **causa** de parada de
emergência (C3 264 ppm em 23/11/2013 e C3 999 ppm em 02/08/2018, ambas com furo confirmado na
abertura) e contra o detector diário sobre as **37 falhas ancoradas** (números recalculados em
11/08/2026, após o resgate de eventos):

| Tratamento | Leituras-gatilho | Máximo do C3 | Detector max>10 (F2) | Detector mediana≥3.5 (F2) |
|---|---|---|---|---|
| Original + `limpar_excursoes` | **2/2 mantidas** | 999 ppm | **0.176** | 0.209 |
| Intervalo 0-10 | 0/2 | 10.0 ppm | 0.000 | 0.212 |
| IQR | 0/2 | 4.0 ppm | 0.000 | 0.209 |
| Hampel 15d | 0/2 | 6.9 ppm | 0.000 | 0.172 |
| Hampel 90d | 0/2 | 4.7 ppm | 0.000 | 0.134 |

Duas conclusões:

1. **Todo filtro estatístico deleta o alvo.** No IQR o maior valor sobrevivente é 4.0 ppm —
   abaixo do próprio limite operacional de 5 ppm: avaliar a regra vigente sobre dados IQR é
   impossível por construção.
2. **A estatística robusta substitui o filtro.** A mediana diária entrega o mesmo F2 sobre dados
   brutos e filtrados — o filtro não melhora o detector robusto e destrói o detector de cauda.

**Decisão (duas pistas):**

- **Pista A — canônica** (tabelas de evento, features, modelos, detector, baseline): sempre
  `Original` = bruto + `limpar_excursoes`, e `limpar_excursoes` descarta **apenas o valor
  extremo** (acima de 1000 ppm — uma única amostra em toda a base, 20000 ppm no C2). A regra
  causal (corroboração na vizinhança OU parada de amostragem em seguida) continua sendo
  calculada, mas como **rótulo de diagnóstico**, não como filtro: a leitura alta isolada fica
  na série porque é uma ultrapassagem sem falha, ou seja, o **falso positivo** que o modelo
  precisa aprender a rejeitar. Robustez a spike espúrio vem de features robustas (mediana,
  p75/p90) e das features relativas — não de pré-filtro.
- **Pista B — visualização e EDA distribucional**: `Intervalo 0-10`, `IQR` e `Hampel` continuam
  existindo para gráficos de série, KDE/violin e comparação descritiva (sem eles, 999 ppm
  esmaga qualquer escala). **Nunca alimentam janela de evento nem modelo.**

As células que comparam tratamentos lado a lado (estatística descritiva, effect size por
tratamento, violins por classe) foram mantidas porque são a evidência dessa decisão.

In [ ]:
# Parâmetros do filtro de Hampel (janela em amostras = dias * amostras por dia)
dias             = 15   # janela curta
dias_longo       = 90   # janela longa, usada para limpar a série completa
amostras_por_dia = 5    # aproximação: a base tem ~6 amostras de laboratório por dia

window_size = dias * amostras_por_dia

## Crystallizer #1

In [ ]:
base_name_crystallizer1 = "Crystallizer #1.csv"

# Linhas 23 e 2141 estão duplicadas mas não possuem amostras significativas (i.e amostras com valores muito baixos)
df_crystallizer1, df_duplicados_crystallizer1 = carregar_crystallizer(
    base_name_crystallizer1, linhas_remover=[23, 2141], limite_ppm=None
)

# limite_ppm=None desliga o corte fixo. `limpar_excursoes` descarta APENAS o valor EXTREMO
# (acima do teto de implausibilidade de 1000 ppm). As leituras altas isoladas continuam na
# série mesmo quando a regra causal as classifica como provável erro de laboratório: uma
# ultrapassagem alta que NÃO terminou em falha é justamente o falso positivo que o modelo
# precisa aprender a rejeitar — e é o mais difícil deles. Ver a seção "Erro de laboratório
# vs medição alta informativa" para o veredito amostra a amostra.
df_crystallizer1, df_descartes_crystallizer1 = limpar_excursoes(
    df_crystallizer1, descartar_isoladas=False
)

tratamentos_crystallizer1, info_crystallizer1 = aplicar_tratamentos(
    df_crystallizer1,
    dias_hampel=dias, dias_hampel_longo=dias_longo, amostras_por_dia=amostras_por_dia
)

df_crystallizer1

### Removendo outliers 0 a 10 #1

In [ ]:
df_crystallizer1_0a10 = tratamentos_crystallizer1["Intervalo 0-10"]

print(f"Amostras originais : {len(df_crystallizer1)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer1_0a10)} ({len(df_crystallizer1) - len(df_crystallizer1_0a10)} removidas)")

df_crystallizer1_0a10

### Removendo outliers com IQR #1

In [ ]:
df_crystallizer1_iqr = tratamentos_crystallizer1["IQR"]
limite_inf, limite_sup = info_crystallizer1["limites_iqr"]

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer1)}")
print(f"Amostras removidas: {len(df_crystallizer1) - len(df_crystallizer1_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer1_iqr)}")

df_crystallizer1_iqr

### Removendo outliers com Filtro de Hampel #1

In [ ]:
df_crystallizer1_hampel = tratamentos_crystallizer1["Hampel"]

print(f"Janela: {dias} dias ({dias * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer1['outliers_hampel']}")
df_crystallizer1_hampel

#### Removendo outliers com Filtro de Hampel window_size=90 dias #1

In [ ]:
df_crystallizer1_hampel90d = tratamentos_crystallizer1["Hampel 90d"]

print(f"Janela: {dias_longo} dias ({dias_longo * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer1['outliers_hampel_longo']}")
df_crystallizer1_hampel90d

## Crystallizer #2

In [ ]:
base_name_crystallizer2 = "Crystallizer #2.csv"

# A amostra de 20000 ppm (Labref 4027521) é o único descarte da base inteira — cai no teto
# de implausibilidade. A segunda maior leitura de toda a base é 999 ppm, confirmada por furo.
df_crystallizer2, df_duplicados_crystallizer2 = carregar_crystallizer(
    base_name_crystallizer2, limite_ppm=None
)

# limite_ppm=None desliga o corte fixo. `limpar_excursoes` descarta APENAS o valor EXTREMO
# (acima do teto de implausibilidade de 1000 ppm). As leituras altas isoladas continuam na
# série mesmo quando a regra causal as classifica como provável erro de laboratório: uma
# ultrapassagem alta que NÃO terminou em falha é justamente o falso positivo que o modelo
# precisa aprender a rejeitar — e é o mais difícil deles. Ver a seção "Erro de laboratório
# vs medição alta informativa" para o veredito amostra a amostra.
df_crystallizer2, df_descartes_crystallizer2 = limpar_excursoes(
    df_crystallizer2, descartar_isoladas=False
)

tratamentos_crystallizer2, info_crystallizer2 = aplicar_tratamentos(
    df_crystallizer2,
    dias_hampel=dias, dias_hampel_longo=dias_longo, amostras_por_dia=amostras_por_dia
)

df_crystallizer2

### Removendo outliers 0 a 10 #2

In [ ]:
df_crystallizer2_0a10 = tratamentos_crystallizer2["Intervalo 0-10"]

print(f"Amostras originais : {len(df_crystallizer2)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer2_0a10)} ({len(df_crystallizer2) - len(df_crystallizer2_0a10)} removidas)")

df_crystallizer2_0a10

### Removendo outliers com IQR #2

In [ ]:
df_crystallizer2_iqr = tratamentos_crystallizer2["IQR"]
limite_inf, limite_sup = info_crystallizer2["limites_iqr"]

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer2)}")
print(f"Amostras removidas: {len(df_crystallizer2) - len(df_crystallizer2_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer2_iqr)}")

df_crystallizer2_iqr

### Removendo outliers com Filtro de Hampel #2

In [ ]:
df_crystallizer2_hampel = tratamentos_crystallizer2["Hampel"]

print(f"Janela: {dias} dias ({dias * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer2['outliers_hampel']}")
df_crystallizer2_hampel

#### Removendo outliers com Filtro de Hampel window_size=90 dias #2

In [ ]:
df_crystallizer2_hampel90d = tratamentos_crystallizer2["Hampel 90d"]

print(f"Janela: {dias_longo} dias ({dias_longo * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer2['outliers_hampel_longo']}")
df_crystallizer2_hampel90d

## Crystallizer #3

In [ ]:
base_name_crystallizer3 = "Crystallizer #3.csv"

# Linhas 6476 e 6494 estão duplicadas mas não possuem amostras significativas
df_crystallizer3, df_duplicados_crystallizer3 = carregar_crystallizer(
    base_name_crystallizer3, linhas_remover=[6476, 6494], limite_ppm=None
)

# limite_ppm=None desliga o corte fixo. `limpar_excursoes` descarta APENAS o valor EXTREMO
# (acima do teto de implausibilidade de 1000 ppm). As leituras altas isoladas continuam na
# série mesmo quando a regra causal as classifica como provável erro de laboratório: uma
# ultrapassagem alta que NÃO terminou em falha é justamente o falso positivo que o modelo
# precisa aprender a rejeitar — e é o mais difícil deles. Ver a seção "Erro de laboratório
# vs medição alta informativa" para o veredito amostra a amostra.
df_crystallizer3, df_descartes_crystallizer3 = limpar_excursoes(
    df_crystallizer3, descartar_isoladas=False
)

tratamentos_crystallizer3, info_crystallizer3 = aplicar_tratamentos(
    df_crystallizer3,
    dias_hampel=dias, dias_hampel_longo=dias_longo, amostras_por_dia=amostras_por_dia
)

df_crystallizer3

In [ ]:
# Esperado: VAZIO. Nenhuma amostra do C3 passa do teto de 1000 ppm — inclusive as leituras
# de 264, 268/366/488 e 999 ppm ficam na base. O único descarte de toda a base é a amostra
# de 20000 ppm do C2 (df_descartes_crystallizer2).
df_descartes_crystallizer3

### Removendo outliers 0 a 10 #3

In [ ]:
df_crystallizer3_0a10 = tratamentos_crystallizer3["Intervalo 0-10"]

print(f"Amostras originais : {len(df_crystallizer3)}")
print(f"Amostras no intervalo [0, 10]: {len(df_crystallizer3_0a10)} ({len(df_crystallizer3) - len(df_crystallizer3_0a10)} removidas)")

df_crystallizer3_0a10

### Removendo outliers com IQR #3

In [ ]:
df_crystallizer3_iqr = tratamentos_crystallizer3["IQR"]
limite_inf, limite_sup = info_crystallizer3["limites_iqr"]

print(f"Limites IQR       : [{limite_inf:.3f}, {limite_sup:.3f}]")
print(f"Amostras originais: {len(df_crystallizer3)}")
print(f"Amostras removidas: {len(df_crystallizer3) - len(df_crystallizer3_iqr)}")
print(f"Amostras restantes: {len(df_crystallizer3_iqr)}")

df_crystallizer3_iqr

### Removendo outliers com Filtro de Hampel #3

In [ ]:
df_crystallizer3_hampel = tratamentos_crystallizer3["Hampel"]

print(f"Janela: {dias} dias ({dias * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer3['outliers_hampel']}")
df_crystallizer3_hampel

#### Removendo outliers com Filtro de Hampel window_size=90 dias #3

In [ ]:
df_crystallizer3_hampel90d = tratamentos_crystallizer3["Hampel 90d"]

print(f"Janela: {dias_longo} dias ({dias_longo * amostras_por_dia} amostras)")
print(f"Outliers detectados: {info_crystallizer3['outliers_hampel_longo']}")
df_crystallizer3_hampel90d

# Erro de laboratório vs medição alta informativa

O corte fixo de 100 ppm que existia antes descartava as duas leituras que a planilha de inspeção registra como **causa** de parada de emergência (264 ppm em 23/11/2013 e 999 ppm em 02/08/2018, ambas no C3, ambas com furo confirmado na abertura).

## Política atual: só o valor EXTREMO é descartado

`limpar_excursoes` remove **uma única classe de amostra**: a que passa do **teto de implausibilidade** (1000 ppm). Na base inteira existe exatamente uma — 20000 ppm no C2, 05/02/2019 — e a segunda maior leitura de toda a base é 999 ppm, confirmada por furo. Todo o resto permanece na série.

A regra causal (`classificar_amostras_altas`) continua rodando, mas agora é **diagnóstico, não filtro**. Ela responde "essa leitura alta tem cara de excursão real ou de leitura isolada?" através de dois critérios:

1. **corroboração** — pelo menos 2 outras amostras acima de 5 ppm em ±2 dias;
2. **seguida de parada** — a próxima amostra normal só aparece mais de 2 dias depois (numa falha abrupta a operação para o reator logo após a leitura, então a excursão não chega a aparecer em outras amostras — foi o caso de 23/11/2013 no C3).

### Por que a leitura alta isolada NÃO é mais descartada

Uma leitura isolada acima de 20 ppm é uma **ultrapassagem que não terminou em falha** — ou seja, é exatamente o falso positivo que o modelo tem de aprender a rejeitar, e é o mais difícil deles, porque é o de maior amplitude. Apagá-la fazia três estragos:

- **retirava da base o exemplo negativo mais informativo** (são 13 amostras entre 21 e 37 ppm, distribuídas em C1 5, C2 2, C3 6);
- **inflava a precisão dos detectores de cauda por construção**: `max > 20 ppm` ia de 14 para 5 falsos positivos e de 0.222 para 0.375 de precisão sem que o detector tivesse melhorado em nada — o ganho vinha de o filtro ter apagado justamente os casos que o detector erra;
- **descartava pelo menos um caso informativo**: a leitura de 23.7 ppm no C1 em 20/05/2011, classificada como isolada, está **8.5 dias antes** de uma falha ancorada. Mantê-la faz `max > 10 ppm` subir de 6 para 7 verdadeiros positivos (F2 0.181 → 0.200).

Efeito de manter tudo (só o teto): **13 amostras a mais** na base, **+4 eventos Real=0** no LC 10 ppm (25 → 29) e **+1** no LC 5 ppm (103 → 104). As 31 falhas ancoradas não mudam.

Robustez a spike espúrio passa a vir de onde deve vir — de **feature robusta** (mediana diária, p75/p90) e das features relativas — e não de apagar amostra. A tabela abaixo mostra o veredito de cada leitura alta; nenhuma delas sai da base.

In [ ]:
# Todas as amostras acima de 20 ppm e o veredito da regra causal.
# ATENÇÃO: o veredito é DIAGNÓSTICO — nenhuma destas amostras é removida da base.
# "isolada" NÃO significa "erro descartado": significa candidata a falso positivo,
# que é precisamente o que a classe Real=0 do modelo precisa conter.
LC_ALTO = 20

relatorio_excursoes = []
for nome, df in [("C1", df_crystallizer1), ("C2", df_crystallizer2), ("C3", df_crystallizer3)]:
    c = classificar_amostras_altas(df, lc_alto=LC_ALTO)
    alta = c[c["Alta"]].copy()
    alta["Crystallizer"] = nome
    relatorio_excursoes.append(alta)

relatorio_excursoes = pd.concat(relatorio_excursoes, ignore_index=True)
relatorio_excursoes["Veredito"] = np.where(
    relatorio_excursoes["ExcursaoReal"], "excursão real", "isolada (candidata a falso positivo)"
)
relatorio_excursoes = relatorio_excursoes.sort_values(["Crystallizer", "TIMESTAMP"])

print(relatorio_excursoes["Veredito"].value_counts().to_string())
print(f"Todas mantidas na base (descarte só acima de 1000 ppm): {len(relatorio_excursoes)}")

relatorio_excursoes[["Crystallizer", "TIMESTAMP", "Resultado de Ferro (ppm)",
                     "Corroborada", "SeguidaDeParada", "Veredito"]]

In [ ]:
# O que REALMENTE saiu da base — só o extremo acima do teto de implausibilidade
for nome, desc in [("C1", df_descartes_crystallizer1),
                   ("C2", df_descartes_crystallizer2),
                   ("C3", df_descartes_crystallizer3)]:
    print(f"{nome}: {len(desc)} amostra(s) descartada(s)")
    if len(desc):
        print(desc[["TIMESTAMP", "Resultado de Ferro (ppm)", "Motivo"]].to_string(index=False))
    print()

# Contraprova da decisão: o que a política antiga (descartar as leituras isoladas) tiraria.
# Cada uma dessas amostras é uma ultrapassagem sem falha — matéria-prima da classe Real=0.
print("-" * 78)
print("Leituras altas isoladas MANTIDAS de propósito (candidatas a falso positivo):")
for nome, df in [("C1", df_crystallizer1), ("C2", df_crystallizer2), ("C3", df_crystallizer3)]:
    c = classificar_amostras_altas(df, lc_alto=LC_ALTO)
    iso = c[c["ErroLab"]]
    faixa = (f"{iso['Resultado de Ferro (ppm)'].min():.1f}-{iso['Resultado de Ferro (ppm)'].max():.1f} ppm"
             if len(iso) else "-")
    print(f"  {nome}: {len(iso):>2} amostra(s)  ({faixa})")

# Carregando eventos identificados

## Deifinindo LC

In [ ]:
# LC usado para sintetizar a classe negativa (Real=0): ultrapassagens sem falha relatada.
# 10 ppm (e não os 5 ppm operacionais) torna o negativo "difícil": uma excursão clara que
# mesmo assim não terminou em falha. A variante no LC operacional de 5 ppm — o cenário real
# de alarme, com ~3x mais negativos — é construída na seção "Unificando eventos".
threshold = 10 #10 #5 #7

## Planilha de inspeção e tabela de eventos

Os eventos vêm de `data/Vitrificados do PIA - Dados de inspeção.xlsx` (uma aba por reator).

### A âncora vem da planilha (coluna "DATA ANCORADA NA SÉRIE DE FERRO")

Essa coluna é a **fonte de verdade da data de cada evento** — é ela que define o instante usado
para recortar a janela `[ts − dias, ts)` de todas as análises. Ela traz três tipos de conteúdo, e
o carregamento trata cada um:

| Conteúdo | Quantas | O que o pipeline faz |
|---|---|---|
| **data e hora** (ex.: `2013-11-23 12:03`) | 56 | vira a âncora do evento |
| `Fora do Período da Série` | 41 | texto → sem âncora; são apontamentos anteriores ao início da série, já excluídos pelo filtro de período |
| `-` | 4 | texto → sem âncora; cai na reancoragem automática |

**As âncoras são timestamps de medição, truncados ao minuto**: 54 das 56 coincidem com uma amostra
real da série a menos de 60 segundos de distância. Por isso o pipeline **casa a âncora com a
medição correspondente** (tolerância de 5 minutos) e fixa o evento **um segundo depois dela** — a
amostra que disparou a parada precisa cair *dentro* da janela, não fora. Nos dois casos sem hora
útil (célula com `00:00`), usa-se a última medição até o fim daquele dia.

**Guarda contra erro de digitação.** Uma âncora a mais de 180 dias da data do apontamento é
ignorada e reportada em tela. Hoje ela pega **um caso**: o evento de **07-08/06/2016 do C1 está
ancorado em `2018-06-05`** — 728 dias depois. O horário (`10:18`) é exatamente o da âncora que o
método automático encontra em **2016**-06-05, ou seja, é o ano que foi digitado errado. Enquanto a
planilha não for corrigida, esse evento usa a âncora automática.

### As demais etapas

- **reancoragem automática (fallback)** — para as linhas sem âncora manual: quando o reator para, a
  amostragem para junto, e o apontamento cai no meio do período sem medição; o evento é deslocado
  para a última medição antes da lacuna;
- **parada de planta vs parada de reator** — lacunas simultâneas nos três reatores são parada de
  planta, não falha de equipamento. A regra é seletiva: **dano físico real (`Vazamento`) é
  resgatado** — o furo encontrado na abertura já existia antes da parada — e a troca **sem** dano
  (preventiva/spare, como C3 05/2020) continua fora. Os resgatados carregam `DescobertoEmParada`;
- **fusão de apontamentos** — a planilha registra a mesma parada em mais de uma linha; apontamentos
  do mesmo reator a menos de 7 dias viram um só, e os **marcadores do registro absorvido são
  propagados** (`Emergencia`, `Vazamento`, `TrocaDoReator`, `FerroCitado`), senão o evento
  sobrevivente perderia o rótulo "emergência" só por ser o mais antigo dos dois;
- **marcadores derivados do texto + correções documentadas** — regex sobre o texto livre, ampliada
  pela auditoria (`"substituir o reator"`, `"poro passante"`, `"até a parte metálica"`,
  `"infiltraç"`, o typo `"emegência"`), mais as duas correções de `CORRECOES_INSPECAO`.

### O que a auditoria e a coluna de âncora mudaram (31 → 36 falhas)

A auditoria de 11/08/2026 recuperou 6 falhas que os marcadores de texto perdiam:

| Evento | O que era | Por que entra |
|---|---|---|
| C1 04/05/2015 | `Falha=False` | *"poro profundo no revestimento de vidro **até a parte metálica**"* — vidro rompido até o aço |
| C2 12/08/2012 | `Falha=False` | *"O reator operou **apenas 308 dias**"* + R.I. *"para **substituir o reator**"* |
| C2 16/02/2016 | `Falha=False` | 4 pontos de dano no revestimento; *"**Substituido o reator**, eixo, baffle..."* |
| C3 28/07/2023 | descartado (parada de planta) | *"Plug de Tântalo instalado no local do **furo do reator**"* |
| C3 18/07/2024 | descartado (parada de planta) | reparo existente *"apresentava **infiltração**"* |
| C3 22/10/2024 | `Falha=False` | o deck da Bayer chama de *"Vazamento no plug do reparo"* — correção documentada |

E a coluna de âncora fundiu um par: **C1 04/05/2015 e 13/05/2015 receberam a mesma âncora**
(`2015-05-04 04:07`), ou seja, a planta os trata como **um único episódio** — a constatação do poro
e a parada de emergência que veio dela. Com isso o total fica em **36 falhas (C1 11, C2 16, C3 9)**,
e as **pós-2019 continuam em 6** (contra 3 antes da auditoria).

In [ ]:
MAP_MEDICOES_BASE = {
    "C1": df_crystallizer1,
    "C2": df_crystallizer2,
    "C3": df_crystallizer3,
}

df_inspecoes_bruto = carregar_inspecoes()
paradas_de_planta  = identificar_paradas_de_planta(MAP_MEDICOES_BASE)

print(f"Paradas de planta identificadas (lacuna simultânea nos 3 reatores): {len(paradas_de_planta)}")
for inicio, fim in paradas_de_planta:
    print(f"   {inicio:%d/%m/%Y} -> {fim:%d/%m/%Y}  ({(fim - inicio).days} dias)")

In [ ]:
# A classificação do modo de falha entra já aqui para viajar junto com a tabela de
# eventos e com a ficha de validação da planta (é uma das colunas que eles vão confirmar).
df_inspecoes = classificar_modo_falha(
    montar_tabela_eventos(df_inspecoes_bruto, MAP_MEDICOES_BASE, paradas_de_planta)
)

In [ ]:
# Âncoras da planilha ("DATA ANCORADA NA SÉRIE DE FERRO") e fallback automático
manuais = df_inspecoes[df_inspecoes["NoPeriodo"] & df_inspecoes["AncoraManual"]]
casaram = int(manuais["AncoraCasouMedicao"].sum())
print(f"Eventos com âncora manual: {len(manuais)}  "
      f"({casaram} casaram com uma medição real; {len(manuais) - casaram} usaram o fim do dia)")

rejeitadas = df_inspecoes[df_inspecoes["AncoraRejeitada"]]
if len(rejeitadas):
    print(f"\nÂNCORAS REJEITADAS pela guarda de 180 dias (provável erro de digitação): {len(rejeitadas)}")
    print(rejeitadas[["Crystallizer", "Inicio", "DataAncoradaManual", "TS_Ajustado"]].to_string(index=False))

deslocados = df_inspecoes[df_inspecoes["NoPeriodo"] & df_inspecoes["Deslocado"]
                          & ~df_inspecoes["AncoraManual"]]
print(f"\nReancorados automaticamente por lacuna (sem âncora na planilha): {len(deslocados)}")

print("\nFalhas selecionadas e a origem de cada âncora:")
df_inspecoes[df_inspecoes["Selecionado"]][
    ["Crystallizer", "Inicio", "DataAncoradaManual", "TS_Ajustado", "AncoraManual",
     "AncoraCasouMedicao", "ApontamentosFundidos", "TipoFalha"]
].sort_values(["Crystallizer", "TS_Ajustado"])

### As datas do deck, da planilha e da âncora — a reancoragem reconcilia

O deck da equipe Bayer e a planilha divergem de data em três eventos. A âncora na série
resolve dois deles — evidência de que a estratégia de ancoragem recupera a data operacional:

| Evento | Deck | Planilha | Âncora na série | Leitura |
|---|---|---|---|---|
| Vazamento bocal do fundo (C1) | 19/01/2024 | 02/02/2024 | **20/01/2024** | a amostragem do C1 para em 20/01 e só volta em 04/02 — a âncora confirma a data do **deck**; a planilha registrou a abertura |
| Vazamento no plug do reparo (C3) | 18/10/2024 | 22/10/2024 | **20/10/2024** | âncora a 2 dias de cada fonte |
| Substituição da BV (C1) | 26/11/2017 | *sem registro* | — | não há apontamento na planilha num raio de 45 dias; as lacunas do C1 próximas são a parada geral de 12–27/10 e 17–21/12. Segue em aberto na ficha de validação |

In [ ]:
# As falhas resgatadas pela auditoria e a qualidade da âncora de cada uma
resgatadas = df_inspecoes[df_inspecoes["Selecionado"] &
                          (df_inspecoes["Corrigido"] | df_inspecoes["DescobertoEmParada"] |
                           (df_inspecoes["TipoFalha"] == "constatada/programada"))]

cols = ["Crystallizer", "Inicio", "TS_Ajustado", "DiasDeslocado", "TipoFalha",
        "DescobertoEmParada", "Corrigido", "Ocorrimento"]
resgatadas[cols].sort_values(["Crystallizer", "Inicio"])

## Crystallizer #1

In [ ]:
df_eventos_crystallizer1 = eventos_para_notebook(df_inspecoes, "C1")

print(f"Falhas de reator no C1: {len(df_eventos_crystallizer1)}")
df_eventos_crystallizer1

### Adicionando momentos em que houve ultrapassagem LC
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

# Atenção: esta célula acrescenta eventos ao df existente — reexecutá-la sem recarregar
# a célula anterior duplica os eventos Real=0
df_eventos_crystallizer1 = adicionar_eventos_ultrapassagem(
    df_crystallizer1, df_eventos_crystallizer1, threshold, dias_baseline=DIAS_BASELINE
)

df_eventos_crystallizer1

### Violin Plot - Testes com diferentes hiperparâmetros
Limpar serie toda com hampel = 90 dias de janela

Ultrapassagem de 10/7 ppm, com hampel de 15 dias

> **Decisão:** a comparação com uma curva normal sintética que existia aqui foi removida.
> A série tem assimetria forte e cauda pesada, então uma normal ajustada por média/desvio
> não é uma referência válida — a referência honesta é a própria série limpa com Hampel 90d.

In [ ]:
# JANELAS = [15, 12, 9, 6, 3]
JANELAS = [15, 7]

metodos = [
    # {"titulo": "Original",       "df": df_crystallizer1},
    # {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    # {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

# Referência: série inteira limpa com Hampel de 90 dias
valores_serie = df_crystallizer1_hampel90d["Resultado de Ferro (ppm)"].dropna().tolist()

fig = plot_violin_serie_vs_ultrapassagem(
    df_eventos_crystallizer1, metodos, JANELAS,
    titulo=(f"Violin Plot: Série Completa vs Janelas de Ultrapassagem do LC ({threshold}ppm) — Crystallizer #1"
            "<br> <sup>Esquerda (azul): distribuição Hampel 90d  |  "
            "Direita (vermelho): distribuição nos N dias antes de qualquer ultrapassagem</sup>"),
    valores_referencia=valores_serie,
)
fig.show()

In [ ]:
# fig.write_html("ViolinPlot_Cristallyzer#1.html")

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer1_filtrado_until2020, df_eventos_crystallizer1_filtrado_after2020 = \
    separar_eventos_por_data(df_eventos_crystallizer1, "2020-04-01")

print(f"Até 01/04/2020 : {len(df_eventos_crystallizer1_filtrado_until2020)} eventos")
print(f"Após 01/04/2020: {len(df_eventos_crystallizer1_filtrado_after2020)} eventos")

df_eventos_crystallizer1_filtrado_until2020.head()

## Crystallizer #2

In [ ]:
df_eventos_crystallizer2 = eventos_para_notebook(df_inspecoes, "C2")

print(f"Falhas de reator no C2: {len(df_eventos_crystallizer2)}")
df_eventos_crystallizer2

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

# Atenção: esta célula acrescenta eventos ao df existente — reexecutá-la sem recarregar
# a célula anterior duplica os eventos Real=0
df_eventos_crystallizer2 = adicionar_eventos_ultrapassagem(
    df_crystallizer2, df_eventos_crystallizer2, threshold, dias_baseline=DIAS_BASELINE
)

df_eventos_crystallizer2

### Violin Plot - Testes com diferentes hiperparâmetros
Limpar serie toda com hampel = 90 dias de janela

Ultrapassagem de 10/7 ppm, com hampel de 15 dias

> **Decisão:** a comparação com uma curva normal sintética que existia aqui foi removida.
> A série tem assimetria forte e cauda pesada, então uma normal ajustada por média/desvio
> não é uma referência válida — a referência honesta é a própria série limpa com Hampel 90d.

In [ ]:
# JANELAS = [15, 12, 9, 6, 3]
JANELAS = [15, 7]

metodos = [
    # {"titulo": "Original",       "df": df_crystallizer2},
    # {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    # {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

# Referência: série inteira limpa com Hampel de 90 dias
valores_serie = df_crystallizer2_hampel90d["Resultado de Ferro (ppm)"].dropna().tolist()

fig = plot_violin_serie_vs_ultrapassagem(
    df_eventos_crystallizer2, metodos, JANELAS,
    titulo=(f"Violin Plot: Série Completa vs Janelas de Ultrapassagem do LC ({threshold}ppm) — Crystallizer #2"
            "<br> <sup>Esquerda (azul): distribuição Hampel 90d  |  "
            "Direita (vermelho): distribuição nos N dias antes de qualquer ultrapassagem</sup>"),
    valores_referencia=valores_serie,
)
fig.show()

In [ ]:
# fig.write_html("ViolinPlot_Cristallyzer#2.html")

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer2_filtrado_until2020, df_eventos_crystallizer2_filtrado_after2020 = \
    separar_eventos_por_data(df_eventos_crystallizer2, "2020-04-01")

print(f"Até 01/04/2020 : {len(df_eventos_crystallizer2_filtrado_until2020)} eventos")
print(f"Após 01/04/2020: {len(df_eventos_crystallizer2_filtrado_after2020)} eventos")

df_eventos_crystallizer2_filtrado_until2020.head()

## Crystallizer #3

In [ ]:
df_eventos_crystallizer3 = eventos_para_notebook(df_inspecoes, "C3")

print(f"Falhas de reator no C3: {len(df_eventos_crystallizer3)}")
df_eventos_crystallizer3

### Adicionando momentos em que houve ultrapassagem do limite de 5ppm
Amostras acima do limite mas que não foram indicadas como problema, a ideia é que a partir de uma ultrapassagem do limiar seja analisada uma janela anterior a essa ultrapassagem para verificar se há de fato um problema no reator

In [ ]:
DIAS_BASELINE = 15

# Atenção: esta célula acrescenta eventos ao df existente — reexecutá-la sem recarregar
# a célula anterior duplica os eventos Real=0
df_eventos_crystallizer3 = adicionar_eventos_ultrapassagem(
    df_crystallizer3, df_eventos_crystallizer3, threshold, dias_baseline=DIAS_BASELINE
)

df_eventos_crystallizer3

### Violin Plot - Testes com diferentes hiperparâmetros
Limpar serie toda com hampel = 90 dias de janela

Ultrapassagem de 10/7 ppm, com hampel de 15 dias

> **Decisão:** a comparação com uma curva normal sintética que existia aqui foi removida.
> A série tem assimetria forte e cauda pesada, então uma normal ajustada por média/desvio
> não é uma referência válida — a referência honesta é a própria série limpa com Hampel 90d.

In [ ]:
# JANELAS = [15, 12, 9, 6, 3]
JANELAS = [15, 7]

metodos = [
    # {"titulo": "Original",       "df": df_crystallizer3},
    # {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    # {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

# Referência: série inteira limpa com Hampel de 90 dias
valores_serie = df_crystallizer3_hampel90d["Resultado de Ferro (ppm)"].dropna().tolist()

fig = plot_violin_serie_vs_ultrapassagem(
    df_eventos_crystallizer3, metodos, JANELAS,
    titulo=(f"Violin Plot: Série Completa vs Janelas de Ultrapassagem do LC ({threshold}ppm) — Crystallizer #3"
            "<br> <sup>Esquerda (azul): distribuição Hampel 90d  |  "
            "Direita (vermelho): distribuição nos N dias antes de qualquer ultrapassagem</sup>"),
    valores_referencia=valores_serie,
)
fig.show()

In [ ]:
# fig.write_html("ViolinPlot_Cristallyzer#3.html")

### Separando eventos até 01/04/2020 e após 01/04/2020

In [ ]:
df_eventos_crystallizer3_filtrado_until2020, df_eventos_crystallizer3_filtrado_after2020 = \
    separar_eventos_por_data(df_eventos_crystallizer3, "2020-04-01")

print(f"Até 01/04/2020 : {len(df_eventos_crystallizer3_filtrado_until2020)} eventos")
print(f"Após 01/04/2020: {len(df_eventos_crystallizer3_filtrado_after2020)} eventos")

df_eventos_crystallizer3_filtrado_until2020.head()

# Baseline: o que a regra vigente entrega

A Etapa 2 tem como objetivo superar o modelo univariado em uso (patamar fixo de ferro). Para
afirmar que algo o supera é preciso primeiro medi-lo — e medir contra a **taxa base**, não contra
zero.

## O que é o lift, e por que ele é a régua deste estudo

Um critério de alarme ("máximo da janela acima de X ppm") pode disparar antes de falha por dois
motivos muito diferentes: porque carrega informação, ou simplesmente porque **dispara o tempo
todo**. Recall sozinho não separa as duas coisas — um critério que dispara em qualquer janela tem
recall alto por definição. O lift separa:

```
lift = P(critério dispara | janela antecede falha) / P(critério dispara | janela qualquer)
```

Na prática: medimos a fração das janelas de 15 dias **anteriores às falhas ancoradas** em que o
critério dispara (numerador) e a mesma fração sobre **2000 janelas sorteadas ao acaso** na série
de cada reator (denominador — a taxa base).

Como ler o número:

- **lift ≈ 1** — o critério dispara antes de falha na mesma proporção em que dispara em qualquer
  momento. **Não informa nada**, por melhor que pareça o recall. É o caso da regra vigente de
  5 ppm (e abaixo de 1 significa que dispara *menos* antes de falha que no dia a dia);
- **lift = 2** — a janela pré-falha tem o dobro da chance de conter o disparo. Informação real,
  mas ainda com muitos falsos positivos se a taxa base for alta;
- **lift alto com taxa base minúscula** (ex.: `max > 20 ppm`, lift ~5x com taxa base ~2%) — o
  critério raramente fala, mas quando fala a chance de falha é várias vezes maior: é o corner de
  alta precisão, útil para o operador.

Três cuidados de leitura, todos deliberados aqui:

1. **lift não é precisão.** Ele compara proporções de janelas, não conta alarmes; um critério de
   lift 5x ainda pode ter precisão baixa se falha é rara (e é — a taxa base de janela pré-falha é
   ~3%). Por isso o baseline reporta também precisão/recall/F2 na tabela de detector, logo abaixo;
2. **a janela importa.** O mesmo critério tem lift diferente em 15 e em 60 dias — se o lift
   desaba quando a janela cresce, o "sinal" está colado no evento (é gatilho, não precursor);
3. **as janelas sorteadas se sobrepõem**, então a taxa base é bem estimada, mas os intervalos de
   confiança de lift seriam otimistas — usamos o lift como régua de comparação entre critérios,
   não como teste de significância.

Todos os resultados do estudo (regra composta, margem cross-reator, modelos) são comparados
contra esta régua, nunca contra zero.

In [ ]:
MAP_EVENTOS_FALHA = {
    "C1": df_eventos_crystallizer1[df_eventos_crystallizer1["Real"] == 1],
    "C2": df_eventos_crystallizer2[df_eventos_crystallizer2["Real"] == 1],
    "C3": df_eventos_crystallizer3[df_eventos_crystallizer3["Real"] == 1],
}

def formatar_lift(df):
    saida = df.copy()
    saida["nas_falhas"] = (100 * saida["nas_falhas"]).round(1).astype(str) + "%"
    saida["taxa_base"]  = (100 * saida["taxa_base"]).round(1).astype(str) + "%"
    saida["lift"]       = saida["lift"].round(2).astype(str) + "x"
    return saida

df_lift = avaliar_baseline_lift(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, janela_dias=15)
formatar_lift(df_lift)

In [ ]:
# Mesma comparação com janela de 60 dias — o limite de 5 ppm fica ABAIXO da taxa base
df_lift_60 = avaliar_baseline_lift(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, janela_dias=7)
formatar_lift(df_lift_60)

## Detector: precisão, recall e F2 da regra vigente e das alternativas

Alarme = estatística diária acima do limite; alarmes a menos de 15 dias contam como um só; acerto se a falha ocorre em até 15 dias depois do alarme.

A tabela abaixo cobre só critérios de patamar sobre a série de cada reator. A régua final — incluindo a **regra composta** com a margem cross-reator, que é a que a Etapa supervisionada precisa bater — está em "Estrutura não supervisionada / A nova régua", depois que a margem é construída.

In [ ]:
configuracoes = [
    ("max",    5,   1),   # regra vigente
    ("max",    7,   1),
    ("max",    10,  1),
    ("max",    20,  1),
    ("median", 3.0, 1),
    ("median", 3.5, 1),
    ("median", 4.0, 1),
    ("median", 3.5, 2),   # exigindo 2 dias seguidos — mostra que o sinal é impulso, não rampa
]

df_detector = pd.DataFrame([
    avaliar_detector_diario(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA,
                            estatistica=est, limite=lim, dias_seguidos=dias)
    for est, lim, dias in configuracoes
]).sort_values("F2", ascending=False)

df_detector

# Plotando Gráfico das medições
Linhas vermelhas = Eventos relatados  
Linhas azuis = Eventos de ultapassagem sem relatos

## Crystallizer #1

In [ ]:
# Sem eventos falsos
fig_crystallizer1 = plot_crystallizer(
    df_crystallizer1, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1",
    mostrar_falsos=False
)
fig_crystallizer1.show()

In [ ]:
# fig_crystallizer1.write_html("Crystallizer#1.html")

### Crystallizer #1 - Outliers 0 a 10

In [ ]:
# Sem eventos falsos
fig_crystallizer1_0a10 = plot_crystallizer(
    df_crystallizer1_0a10, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1 0a10",
    mostrar_falsos=False
)
fig_crystallizer1_0a10.show()

### Crystallizer #1 - Outliers IQR

In [ ]:
# Sem eventos falsos
fig_crystallizer1_iqr = plot_crystallizer(
    df_crystallizer1_iqr, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1 IQR",
    mostrar_falsos=False
)
fig_crystallizer1_iqr.show()

### Crystallizer #1 - Filtro de Hampel

In [ ]:
# Sem eventos falsos
fig_crystallizer1_hampel = plot_crystallizer(
    df_crystallizer1_hampel, df_eventos_crystallizer1,
    titulo="Fe (ppm) - Crystallizer #1 Hampel",
    mostrar_falsos=False
)
fig_crystallizer1_hampel.show()

## Crystallizer #2

In [ ]:
# Sem eventos falsos
fig_crystallizer2 = plot_crystallizer(
    df_crystallizer2, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2",
    mostrar_falsos=False
)
fig_crystallizer2.show()

In [ ]:
# fig_crystallizer2.write_html("Crystallizer#2.html")

### Crystallizer #2 - Outliers 0 a 10

In [ ]:
# Sem eventos falsos
fig_crystallizer2_0a10 = plot_crystallizer(
    df_crystallizer2_0a10, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2 0a10",
    mostrar_falsos=False
)
fig_crystallizer2_0a10.show()

### Crystallizer #2 - Outliers IQR

In [ ]:
# Sem eventos falsos
fig_crystallizer2_iqr = plot_crystallizer(
    df_crystallizer2_iqr, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2 IQR",
    mostrar_falsos=False
)
fig_crystallizer2_iqr.show()

### Crystallizer #2 - Filtro de Hampel

In [ ]:
# Sem eventos falsos
fig_crystallizer2_hampel = plot_crystallizer(
    df_crystallizer2_hampel, df_eventos_crystallizer2,
    titulo="Fe (ppm) - Crystallizer #2 Hampel",
    mostrar_falsos=False
)
fig_crystallizer2_hampel.show()

## Crystallizer #3

In [ ]:
# Sem eventos falsos
fig_crystallizer3 = plot_crystallizer(
    df_crystallizer3, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3",
    mostrar_falsos=False
)
fig_crystallizer3.show()

In [ ]:
# fig_crystallizer3.write_html("Crystallizer#3.html")

### Crystallizer #3 - Outliers 0 a 10

In [ ]:
# Sem eventos falsos
fig_crystallizer3_0a10 = plot_crystallizer(
    df_crystallizer3_0a10, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3 0a10",
    mostrar_falsos=False
)
fig_crystallizer3_0a10.show()

### Crystallizer #3 - Outliers IQR

In [ ]:
# Sem eventos falsos
fig_crystallizer3_iqr = plot_crystallizer(
    df_crystallizer3_iqr, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3 IQR",
    mostrar_falsos=False
)
fig_crystallizer3_iqr.show()

### Crystallizer #3 - Filtro de Hampel

In [ ]:
# Sem eventos falsos
fig_crystallizer3_hampel = plot_crystallizer(
    df_crystallizer3_hampel, df_eventos_crystallizer3,
    titulo="Fe (ppm) - Crystallizer #3 Hampel",
    mostrar_falsos=False
)
fig_crystallizer3_hampel.show()

## Gráfico com Média Móvel
Verificando se há tendência clara nos dados

## MM Crystallizer #1

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #1 - Outliers 0 a 10

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1_0a10, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1 0a10",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #1 - IQR

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1_iqr, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1 IQR",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #1 - Filtro de Hampel

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer1_hampel, df_eventos_crystallizer1,
    titulo="Fe (ppm) MM - Crystallizer #1 Hampel",
    mostrar_falsos=False
)
fig.show()

## MM Crystallizer #2

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #2 - Outliers 0 a 10

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2_0a10, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2 0a10",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #2 - IQR

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2_iqr, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2 IQR",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #2 - Filtro de Hampel

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer2_hampel, df_eventos_crystallizer2,
    titulo="Fe (ppm) MM - Crystallizer #2 Hampel",
    mostrar_falsos=False
)
fig.show()

## MM Crystallizer #3

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #3 - Outliers 0 a 10

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3_0a10, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3 0a10",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #3 - IQR

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3_iqr, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3 IQR",
    mostrar_falsos=False
)
fig.show()

### MM Crystallizer #3 - Filtro de Hampel

In [ ]:
fig = plot_mm_crystallizer(
    df_crystallizer3_hampel, df_eventos_crystallizer3,
    titulo="Fe (ppm) MM - Crystallizer #3 Hampel",
    mostrar_falsos=False
)
fig.show()

# Estatísticas descritivas de toda série de concentração de Fe

In [ ]:
# Coluna separadora vazia
separador = pd.Series({k: "" for k in ["Contagem","Média","Mediana","Desvio Padrão","Variância",
                                        "Mínimo","Máximo","Amplitude","Q1 (25%)","Q3 (75%)","IQR",
                                        "Assimetria","Curtose"]})

c1 = pd.concat([
    estatisticas(df_crystallizer1["Resultado de Ferro (ppm)"].dropna(),       "C1 Original"),
    estatisticas(df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna(),  "C1 Intervalo 0-10"),
    estatisticas(df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna(),   "C1 IQR"),
    estatisticas(df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna(),"C1 Hampel"),
], axis=1)

c2 = pd.concat([
    estatisticas(df_crystallizer2["Resultado de Ferro (ppm)"].dropna(),       "C2 Original"),
    estatisticas(df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna(),  "C2 Intervalo 0-10"),
    estatisticas(df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna(),   "C2 IQR"),
    estatisticas(df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna(),"C2 Hampel"),
], axis=1)

c3 = pd.concat([
    estatisticas(df_crystallizer3["Resultado de Ferro (ppm)"].dropna(),       "C3 Original"),
    estatisticas(df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna(),  "C3 Intervalo 0-10"),
    estatisticas(df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna(),   "C3 IQR"),
    estatisticas(df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna(),"C3 Hampel"),
], axis=1)

sep = separador.rename("│")

df_comparativo = pd.concat([c1, sep, c2, sep.rename("│"), c3], axis=1).round(4)

# Corrige as colunas separadoras que ficaram com float após o round
df_comparativo["│"]  = ""
df_comparativo["│"] = ""

df_comparativo

In [ ]:
metodos = [
    {
        "titulo": "Original",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Intervalo 0-10",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "IQR",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Hampel",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
]

CORES = {"C1": "#1f77b4", "C2": "#2ca02c", "C3": "#ff7f0e"}

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=2,
    subplot_titles=[
        titulo
        for m in metodos
        for titulo in [f"KDE — {m['titulo']}", f"Violin Plot — {m['titulo']}"]
    ]
)

for row_idx, metodo in enumerate(metodos, start=1):
    for c in metodo["series"]:
        serie = c["serie"]
        nome  = c["nome"]
        cor   = CORES[nome]

        # KDE na escala de densidade natural (área sob a curva = 1)
        kde = gaussian_kde(serie)
        x_range = np.linspace(serie.min(), serie.max(), 1000)
        y_kde = kde(x_range)
        y_kde = y_kde / trapezoid(y_kde, x_range)  # normaliza

        fig.add_trace(go.Scatter(
            x=x_range,
            y=y_kde,
            mode='lines',
            line=dict(color=cor, width=2),
            fill='tozeroy',
            fillcolor=hex_to_rgba(cor, alpha=0.15),
            name=nome,
            legendgroup=nome,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

        # Violin
        fig.add_trace(go.Violin(
            y=serie,
            name=nome,
            marker_color=cor,
            fillcolor=hex_to_rgba(cor, alpha=0.4),
            box_visible=True,
            meanline_visible=True,
            legendgroup=nome,
            showlegend=False
        ), row=row_idx, col=2)

    fig.update_xaxes(title_text="Resultado de Ferro (ppm)", row=row_idx, col=1)
    fig.update_yaxes(title_text="Densidade", row=row_idx, col=1)
    fig.update_yaxes(title_text="ppm", row=row_idx, col=2)

fig.update_layout(
    height=500 * n_metodos,
    template='plotly_white',
    title="Análise Descritiva Comparativa — Resultado de Ferro (ppm)",
)
fig.show()

### Teste de Kruskal-Wallis com Effect Size (η²)

Para amostras grandes (n ~ 30.000), testes estatísticos como o Kruskal-Wallis tendem a rejeitar a hipótese nula mesmo com diferenças praticamente irrelevantes. Por isso o p-value é complementado pelo **eta-quadrado (η²)** que mede a proporção da variação total explicada pelo agrupamento por crystallizer.

| η²        | Interpretação                                      |
|-----------|----------------------------------------------------|
| < 0.01    | Efeito negligenciável — unificação justificada     |
| 0.01–0.06 | Efeito pequeno — unificação provavelmente aceitável|
| 0.06–0.14 | Efeito médio — avaliar com cautela                 |
| > 0.14    | Efeito grande — distribuições substancialmente diferentes |

A decisão de unificar os dados dos três crystallizers deve considerar em conjunto o η², o p-value e a inspeção visual das curvas KDE e violin plots gerados.

In [ ]:
metodos = [
    {
        "titulo": "Original",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Intervalo 0-10",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_0a10["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_0a10["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "IQR",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_iqr["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_iqr["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
    {
        "titulo": "Hampel",
        "series": [
            {"nome": "C1", "serie": df_crystallizer1_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C2", "serie": df_crystallizer2_hampel["Resultado de Ferro (ppm)"].dropna()},
            {"nome": "C3", "serie": df_crystallizer3_hampel["Resultado de Ferro (ppm)"].dropna()},
        ]
    },
]

for metodo in metodos:
    series = [c["serie"].values for c in metodo["series"]]
    nomes  = [c["nome"] for c in metodo["series"]]
    n_total = sum(len(s) for s in series)

    stat_kw, p_kw = kruskal(*series)

    # Eta-quadrado: mede o quanto da variação total é explicada pelo grupo
    # 0.01 = pequeno, 0.06 = médio, 0.14 = grande
    eta2 = (stat_kw - len(series) + 1) / (n_total - len(series))

    print(f"\nMétodo: {metodo['titulo']}")
    print(f"  H = {stat_kw:.4f}  |  p = {p_kw:.6f}  |  η² = {eta2:.4f}")
    if eta2 < 0.01:
        print("  → Efeito negligenciável — unificação justificada mesmo com p < 0.05")
    elif eta2 < 0.06:
        print("  → Efeito pequeno — unificação provavelmente aceitável")
    elif eta2 < 0.14:
        print("  → Efeito médio — avaliar com cautela")
    else:
        print("  → Efeito grande — distribuições substancialmente diferentes")

### Teste de normalidade Q-Q Plot

In [ ]:
# Dataset: Original — testar normalidade sobre a série IQR seria circular (o IQR amputa as
# caudas e a conclusão sai enviesada para "normal"). Ver "Política de tratamento de dados".
df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
# df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

serie = df["Resultado de Ferro (ppm)"].dropna()

# Visualização
fig_normalidade_crystallizer1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Q-Q Plot", "Histograma + Distribuição Normal Teórica"]
)

# Q-Q Plot
qq = stats.probplot(serie, dist="norm")
qq_x = [qq[0][0][0], qq[0][0][-1]]
qq_y = [qq[1][1] + qq[1][0] * qq[0][0][0],
        qq[1][1] + qq[1][0] * qq[0][0][-1]]

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq[0][0], y=qq[0][1],
    mode='markers',
    marker=dict(color='steelblue', size=4, opacity=0.5),
    name='Quantis observados'
), row=1, col=1)

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=qq_x, y=qq_y,
    mode='lines',
    line=dict(color='red', width=2),
    name='Linha normal teórica'
), row=1, col=1)

# Histograma + curva normal teórica
x_range = np.linspace(serie.min(), serie.max(), 300)
y_normal = stats.norm.pdf(x_range, serie.mean(), serie.std())
y_normal_scaled = y_normal * len(serie) * (serie.max() - serie.min()) / 50

fig_normalidade_crystallizer1.add_trace(go.Histogram(
    x=serie, nbinsx=50,
    marker_color='steelblue', opacity=0.6,
    name='Dados observados'
), row=1, col=2)

fig_normalidade_crystallizer1.add_trace(go.Scatter(
    x=x_range, y=y_normal_scaled,
    mode='lines',
    line=dict(color='red', width=2),
    name='Normal teórica'
), row=1, col=2)

fig_normalidade_crystallizer1.update_xaxes(title_text="Quantis teóricos", row=1, col=1)
fig_normalidade_crystallizer1.update_yaxes(title_text="Quantis observados", row=1, col=1)
fig_normalidade_crystallizer1.update_xaxes(title_text="Resultado de Ferro (ppm)", row=1, col=2)
fig_normalidade_crystallizer1.update_yaxes(title_text="Contagem", row=1, col=2)

fig_normalidade_crystallizer1.update_layout(
    height=500,
    template='plotly_white',
    title="Análise de Normalidade — Resultado de Ferro (ppm) Cristallyzer #1"
)
fig_normalidade_crystallizer1.show()

# Unificando bases de dados

## Unificando dados

In [ ]:
# Original
df_crystallizer123 = pd.concat([
    df_crystallizer1.assign(Crystallizer='C1'),
    df_crystallizer2.assign(Crystallizer='C2'),
    df_crystallizer3.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Originais: {df_crystallizer123.shape}")

# Intervalo 0-10
df_crystallizer123_0a10 = pd.concat([
    df_crystallizer1_0a10.assign(Crystallizer='C1'),
    df_crystallizer2_0a10.assign(Crystallizer='C2'),
    df_crystallizer3_0a10.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Intervalo 0-10: {df_crystallizer123_0a10.shape}")

# IQR
df_crystallizer123_iqr = pd.concat([
    df_crystallizer1_iqr.assign(Crystallizer='C1'),
    df_crystallizer2_iqr.assign(Crystallizer='C2'),
    df_crystallizer3_iqr.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"IQR: {df_crystallizer123_iqr.shape}")

# Hampel
df_crystallizer123_hampel = pd.concat([
    df_crystallizer1_hampel.assign(Crystallizer='C1'),
    df_crystallizer2_hampel.assign(Crystallizer='C2'),
    df_crystallizer3_hampel.assign(Crystallizer='C3'),
], ignore_index=True).sort_values('TIMESTAMP')
print(f"Hampel: {df_crystallizer123_hampel.shape}")

## Unificando eventos

In [ ]:
# Eventos unificados — igual para todos os tratamentos.
# A deduplicação cross-reator (remoção de Real=0 perto de qualquer Real=1 e espaçamento
# mínimo entre Real=0) vive em utils.unificar_eventos, usada também pela variante de
# 5 ppm abaixo — as duas tabelas passam exatamente pela mesma regra.
df_eventos_crystallizer123 = unificar_eventos({
    "C1": df_eventos_crystallizer1,
    "C2": df_eventos_crystallizer2,
    "C3": df_eventos_crystallizer3,
}, intervalo_min_dias=15)

## Variante: classe negativa no LC operacional (5 ppm)

O `threshold = 10` gera negativos "difíceis" (excursões claras sem falha), mas a regra vigente
e o custo da condenação indevida vivem em **5 ppm**. Para a Etapa 2 responder "reduzimos os
falsos positivos da regra vigente?", a classe negativa também precisa existir no limiar em que
a planta opera — o que multiplica os negativos e torna o problema mais realista.

As duas tabelas seguem em paralelo: `df_eventos_*` (LC 10 ppm) e `df_eventos_*_lc5` (LC 5 ppm).
Qualquer resultado supervisionado deve ser reportado nas duas.

In [ ]:
LC_OPERACIONAL = 5

# Reconstrói a parte Real=1 direto da planilha de inspeção (eventos_para_notebook) em vez de
# reaproveitar df_eventos_crystallizerN — evita herdar os Real=0 de LC 10 e o risco de
# duplicação por reexecução.
df_eventos_crystallizer1_lc5 = adicionar_eventos_ultrapassagem(
    df_crystallizer1, eventos_para_notebook(df_inspecoes, "C1"),
    LC_OPERACIONAL, dias_baseline=DIAS_BASELINE)
df_eventos_crystallizer2_lc5 = adicionar_eventos_ultrapassagem(
    df_crystallizer2, eventos_para_notebook(df_inspecoes, "C2"),
    LC_OPERACIONAL, dias_baseline=DIAS_BASELINE)
df_eventos_crystallizer3_lc5 = adicionar_eventos_ultrapassagem(
    df_crystallizer3, eventos_para_notebook(df_inspecoes, "C3"),
    LC_OPERACIONAL, dias_baseline=DIAS_BASELINE)

df_eventos_crystallizer123_lc5 = unificar_eventos({
    "C1": df_eventos_crystallizer1_lc5,
    "C2": df_eventos_crystallizer2_lc5,
    "C3": df_eventos_crystallizer3_lc5,
}, intervalo_min_dias=15)

# Comparação das duas classes negativas
comparacao = pd.DataFrame({
    "LC 10 ppm": df_eventos_crystallizer123["Real"].value_counts(),
    "LC 5 ppm (operacional)": df_eventos_crystallizer123_lc5["Real"].value_counts(),
}).rename(index={1: "Real=1 (falhas)", 0: "Real=0 (ultrapassagens)"})
comparacao

## Auditoria da tabela de eventos e ficha para a planta

Duas verificações que faltavam antes de qualquer modelo consumir esta tabela:

1. **suporte da janela** — nas janelas deslizantes o suporte insuficiente é filtrado, mas na
   tabela de eventos um evento com poucas amostras em 15 dias entra calado e vira estatística
   de dois pontos;
2. **ficha de validação** — todo resultado que depende de rótulo repousa nestes timestamps, e
   há divergência conhecida entre o deck da equipe Bayer e a planilha de inspeção em pelo
   menos três eventos. A ficha põe lado a lado data da planilha, data reancorada, deslocamento
   aplicado, motivo e suporte de amostra, **para a planta validar linha a linha**.

In [ ]:
MAP_FALHAS_AUDITORIA = {
    "C1": df_eventos_crystallizer1[df_eventos_crystallizer1["Real"] == 1],
    "C2": df_eventos_crystallizer2[df_eventos_crystallizer2["Real"] == 1],
    "C3": df_eventos_crystallizer3[df_eventos_crystallizer3["Real"] == 1],
}

df_auditoria_janelas = auditar_janelas_eventos(MAP_MEDICOES_BASE, MAP_FALHAS_AUDITORIA)

print()
df_ficha_eventos = ficha_eventos_para_validacao(df_inspecoes, MAP_MEDICOES_BASE)
print(f"Ficha para validação com a planta: {len(df_ficha_eventos)} apontamentos no período")
print(df_ficha_eventos["status"].value_counts().to_string())
# df_ficha_eventos.to_csv(DIR_OUTPUT + "ficha_eventos_para_validacao.csv", sep=";", index=False)
df_ficha_eventos.head(10)

## Violin Plot das classes

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer123},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer123_0a10},
    {"titulo": "IQR",            "df": df_crystallizer123_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer123_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer123, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1#2#3 unificado"
)
fig.show()

## Plot dados unificados

In [ ]:
# Sem eventos falsos
fig = plot_crystallizer_unificado(
    df_crystallizer123, df_eventos_crystallizer123,
    titulo="Fe (ppm) - Crystallizers Unificados (somente eventos reais)",
    mostrar_falsos=False
)
fig.show()

# Estatísticas descritivas dos eventos

## Crystallizer #1

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer1, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
# df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

df_testes = testar_classes_por_janela(df, df_eventos_crystallizer1, JANELAS)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer1.copy()
# df = df_crystallizer1_0a10.copy()
# df = df_crystallizer1_iqr.copy()
# df = df_crystallizer1_hampel.copy()

JANELAS = [15, 12, 9, 6, 3]

df_effect, fig = separabilidade_features(
    df, df_eventos_crystallizer1, JANELAS, titulo="Crystallizer #1"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer1_filtrado_until2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1 — eventos até 01/04/2020"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer1},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer1_0a10},
    {"titulo": "IQR",            "df": df_crystallizer1_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer1_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer1_filtrado_after2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #1 — eventos após 01/04/2020"
)
fig.show()

## Crystallizer #2

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer2, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer2.copy()
# df = df_crystallizer2_0a10.copy()
# df = df_crystallizer2_iqr.copy()
# df = df_crystallizer2_hampel.copy()

df_testes = testar_classes_por_janela(df, df_eventos_crystallizer2, JANELAS)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer2.copy()
# df = df_crystallizer2_0a10.copy()
# df = df_crystallizer2_iqr.copy()
# df = df_crystallizer2_hampel.copy()

JANELAS = [15, 12, 9, 6, 3]

df_effect, fig = separabilidade_features(
    df, df_eventos_crystallizer2, JANELAS, titulo="Crystallizer #2"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer2_filtrado_until2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2 — eventos até 01/04/2020"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer2},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer2_0a10},
    {"titulo": "IQR",            "df": df_crystallizer2_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer2_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer2_filtrado_after2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #2 — eventos após 01/04/2020"
)
fig.show()

## Crystallizer #3

### Análise por ViolinPlot
Agrupando os dados de todos os eventos por janela e comparando com os dados do falso positivo

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer3, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3"
)
fig.show()

#### Teste estatístico entre as duas classes
Quantificar se diferença entre classes é estatisticamente significativa por janela

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer3.copy()
# df = df_crystallizer3_0a10.copy()
# df = df_crystallizer3_iqr.copy()
# df = df_crystallizer3_hampel.copy()

df_testes = testar_classes_por_janela(df, df_eventos_crystallizer3, JANELAS)
df_testes

#### Separabilidade Estatística Feature a Feature por janela
Cálculo do effect size de Mann-Whitney para cada estatística descritiva por janela verificando diretamente qual feature e qual janela têm maior poder discriminativo

In [ ]:
# Dataset: Original — análise de janela de evento roda na Pista A (ver "Política de
# tratamento de dados"): os tratamentos removem justamente max/p90/range, as features
# que separam as classes.
df = df_crystallizer3.copy()
# df = df_crystallizer3_0a10.copy()
# df = df_crystallizer3_iqr.copy()
# df = df_crystallizer3_hampel.copy()

JANELAS = [15, 12, 9, 6, 3]

df_effect, fig = separabilidade_features(
    df, df_eventos_crystallizer3, JANELAS, titulo="Crystallizer #3"
)
fig.show()

### Análise dos eventos ATÉ 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer3_filtrado_until2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3 — eventos até 01/04/2020"
)
fig.show()

### Análise dos eventos APÓS 01/04/2020

#### ViolinPlot

In [ ]:
JANELAS = [15, 12, 9, 6, 3]

metodos = [
    {"titulo": "Original",       "df": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "df": df_crystallizer3_0a10},
    {"titulo": "IQR",            "df": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "df": df_crystallizer3_hampel},
]

fig = plot_violin_classes(
    df_eventos_crystallizer3_filtrado_after2020, metodos, JANELAS,
    titulo="Violin Plot: Real vs Falso Positivo por Janela Temporal — Crystallizer #3 — eventos após 01/04/2020"
)
fig.show()

# Comparação dos 3 reatores

## Verificando effect size por reator

In [ ]:
crystallizers_config = [
    # C1
    (df_crystallizer1,        df_eventos_crystallizer1, "C1 Original"),
    (df_crystallizer1_0a10,   df_eventos_crystallizer1, "C1 0-10"),
    (df_crystallizer1_iqr,    df_eventos_crystallizer1, "C1 IQR"),
    (df_crystallizer1_hampel, df_eventos_crystallizer1, "C1 Hampel"),
    # C2
    (df_crystallizer2,        df_eventos_crystallizer2, "C2 Original"),
    (df_crystallizer2_0a10,   df_eventos_crystallizer2, "C2 0-10"),
    (df_crystallizer2_iqr,    df_eventos_crystallizer2, "C2 IQR"),
    (df_crystallizer2_hampel, df_eventos_crystallizer2, "C2 Hampel"),
    # C3
    (df_crystallizer3,        df_eventos_crystallizer3, "C3 Original"),
    (df_crystallizer3_0a10,   df_eventos_crystallizer3, "C3 0-10"),
    (df_crystallizer3_iqr,    df_eventos_crystallizer3, "C3 IQR"),
    (df_crystallizer3_hampel, df_eventos_crystallizer3, "C3 Hampel"),
    # Unificado
    (df_crystallizer123,        df_eventos_crystallizer123, "Unificado Original"),
    (df_crystallizer123_0a10,   df_eventos_crystallizer123, "Unificado 0-10"),
    (df_crystallizer123_iqr,    df_eventos_crystallizer123, "Unificado IQR"),
    (df_crystallizer123_hampel, df_eventos_crystallizer123, "Unificado Hampel"),
]

STATS_FUNCS = {
    'media'   : np.mean,
    'mediana' : np.median,
    'std'     : np.std,
    'max'     : np.max,
    'p75'     : lambda x: np.percentile(x, 75),
    'p90'     : lambda x: np.percentile(x, 90),
    'skewness': lambda x: float(skew(x)),
    'kurtosis': lambda x: float(kurtosis(x)),
    'range'   : lambda x: np.max(x) - np.min(x),
}

registros_todos = []
for df_c, df_ev, nome_c in crystallizers_config:
    for DIAS_JANELA in JANELAS:
        stat_vals = {s: {0: [], 1: []} for s in STATS_FUNCS}
        for _, evento in df_ev.iterrows():
            ts     = evento["TIMESTAMP"]
            inicio = ts - pd.Timedelta(days=DIAS_JANELA)
            mask   = (df_c['TIMESTAMP'] >= inicio) & (df_c['TIMESTAMP'] < ts)
            v      = df_c[mask]["Resultado de Ferro (ppm)"].dropna().values
            if len(v) < 2:
                continue
            classe = int(evento["Real"])
            for nome_stat, func in STATS_FUNCS.items():
                stat_vals[nome_stat][classe].append(func(v))

        for nome_stat in STATS_FUNCS:
            v0 = np.array(stat_vals[nome_stat][0])
            v1 = np.array(stat_vals[nome_stat][1])
            if len(v0) < 2 or len(v1) < 2:
                continue
            stat_mw, p = mannwhitneyu(v0, v1, alternative='two-sided')
            effect = abs(1 - (2 * stat_mw) / (len(v0) * len(v1)))
            registros_todos.append({
                'crystallizer': nome_c,
                'janela'      : f"{DIAS_JANELA}d",
                'feature'     : nome_stat,
                'effect_size' : round(effect, 4),
                'p_value'     : round(p, 4),
            })

df_effect_todos = pd.DataFrame(registros_todos)

# Heatmap — 4 origens (C1, C2, C3, Unificado) × 4 métodos = 16 subplots
n_cols = 4  # Original, 0-10, IQR, Hampel
n_rows = 4  # C1, C2, C3, Unificado
nomes  = [c[2] for c in crystallizers_config]

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=nomes,
    vertical_spacing=0.06
)

for idx, (_, _, nome_c) in enumerate(crystallizers_config):
    row_idx = idx // n_cols + 1
    col_idx = idx % n_cols + 1

    subset = df_effect_todos[df_effect_todos['crystallizer'] == nome_c]
    pivot  = subset.pivot(index='feature', columns='janela', values='effect_size')
    pivot  = pivot[[f"{d}d" for d in JANELAS]]

    fig.add_trace(go.Heatmap(
        z=pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        colorscale='RdYlGn',
        zmin=0, zmax=1,
        text=np.round(pivot.values, 3),
        texttemplate="%{text}",
        showscale=(col_idx == n_cols and row_idx == n_rows)
    ), row=row_idx, col=col_idx)

fig.update_layout(
    title="Effect Size comparativo — C1, C2, C3 e Unificado × Original, 0-10, IQR, Hampel",
    template="plotly_white",
    height=400 * n_rows
)
fig.show()

## Correlação cruzada entre os crystallizers

In [ ]:
freq = "3D"
pares = [("C1", "C2"), ("C1", "C3"), ("C2", "C3")]

metodos = [
    {"titulo": "Original",       "c1": df_crystallizer1,        "c2": df_crystallizer2,        "c3": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "c1": df_crystallizer1_0a10,   "c2": df_crystallizer2_0a10,   "c3": df_crystallizer3_0a10},
    {"titulo": "IQR",            "c1": df_crystallizer1_iqr,    "c2": df_crystallizer2_iqr,    "c3": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "c1": df_crystallizer1_hampel, "c2": df_crystallizer2_hampel, "c3": df_crystallizer3_hampel},
]

for m in metodos:
    s1 = m["c1"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s2 = m["c2"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s3 = m["c3"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    m["df_corr"] = pd.concat([s1, s2, s3], axis=1, keys=["C1", "C2", "C3"]).dropna()

    print(f"\n{'='*55}")
    print(f"Método: {m['titulo']}")
    print(m["df_corr"].corr(method="pearson").round(3).to_string())

n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=3,
    subplot_titles=[
        f"{a} vs {b} — {m['titulo']}"
        for m in metodos
        for a, b in pares
    ],
    vertical_spacing=0.06
)

for row_idx, m in enumerate(metodos, start=1):
    for col_idx, (a, b) in enumerate(pares, start=1):
        add_scatter_regressao(fig, m["df_corr"][a].values, m["df_corr"][b].values, row=row_idx, col=col_idx)
        fig.update_xaxes(title_text=a, row=row_idx, col=col_idx)
        fig.update_yaxes(title_text=b, row=row_idx, col=col_idx)

fig.update_layout(
    height=400 * n_metodos,
    template='plotly_white',
    title="Correlação cruzada — C1, C2, C3 × Original, 0-10, IQR, Hampel"
)
fig.show()

## Correlação cruzada com lags
Verificando se um reator tem influência sobre outro

In [ ]:
MAX_LAG = 15
lags = range(-MAX_LAG, MAX_LAG + 1)
freq = "3D"
pares = [("C1", "C2"), ("C1", "C3"), ("C2", "C3")]

metodos = [
    {"titulo": "Original",       "c1": df_crystallizer1,        "c2": df_crystallizer2,        "c3": df_crystallizer3},
    {"titulo": "Intervalo 0-10", "c1": df_crystallizer1_0a10,   "c2": df_crystallizer2_0a10,   "c3": df_crystallizer3_0a10},
    {"titulo": "IQR",            "c1": df_crystallizer1_iqr,    "c2": df_crystallizer2_iqr,    "c3": df_crystallizer3_iqr},
    {"titulo": "Hampel",         "c1": df_crystallizer1_hampel, "c2": df_crystallizer2_hampel, "c3": df_crystallizer3_hampel},
]

# Prepara df_corr para cada método
for m in metodos:
    s1 = m["c1"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s2 = m["c2"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    s3 = m["c3"].set_index("TIMESTAMP")["Resultado de Ferro (ppm)"].resample(freq).mean()
    m["df_corr"] = pd.concat([s1, s2, s3], axis=1, keys=["C1", "C2", "C3"]).dropna()

# Cross-correlação com lag
n_metodos = len(metodos)
fig = make_subplots(
    rows=n_metodos, cols=1,
    subplot_titles=[f"Cross-correlação com Defasagem — {m['titulo']}" for m in metodos],
    vertical_spacing=0.08
)

for row_idx, m in enumerate(metodos, start=1):
    df_c = m["df_corr"]
    resultados_lag = []

    for a, b in pares:
        x = df_c[a].values
        y = df_c[b].values
        for lag in lags:
            if lag < 0:
                xs, ys = x[:lag],  y[-lag:]
            elif lag > 0:
                xs, ys = x[lag:],  y[:-lag]
            else:
                xs, ys = x, y
            r, p = pearsonr(xs, ys)
            resultados_lag.append({
                "par": f"{a} vs {b}", "lag_dias": lag * 3,
                "r": round(r, 4), "p": round(p, 6)
            })

    df_lag = pd.DataFrame(resultados_lag)

    for par in df_lag["par"].unique():
        sub = df_lag[df_lag["par"] == par]
        fig.add_trace(go.Scatter(
            x=sub["lag_dias"], y=sub["r"],
            mode="lines", name=par,
            line=dict(width=2),
            legendgroup=par,
            showlegend=(row_idx == 1)
        ), row=row_idx, col=1)

    fig.add_vline(x=0, line_dash="dash", line_color="black")
    fig.add_hline(y=0, line_color="gray", line_width=0.5, row=row_idx, col=1)
    fig.update_yaxes(title_text="Pearson r", row=row_idx, col=1)
    fig.update_xaxes(title_text="Defasagem (dias)", row=row_idx, col=1)

    # Lag de máxima correlação
    print(f"\nMétodo: {m['titulo']} — Lag de máxima correlação:")
    for par in df_lag["par"].unique():
        sub  = df_lag[df_lag["par"] == par]
        best = sub.loc[sub["r"].idxmax()]
        print(f"  {par}: lag={best['lag_dias']:.0f} dias  |  r={best['r']:.4f}")

fig.update_layout(
    height=400 * n_metodos,
    template="plotly_white",
    hovermode="x unified",
    title="Correlação cruzada com Defasagem entre Reatores<br>"
          "<sup>Pico em lag≠0 indica que um reator influencia o outro, negativo: A influencia B  |  positivo: B influencia A</sup>"
)
fig.show()

# Estrutura não supervisionada (regimes de operação)

1. **Janelas deslizantes de toda a série** (~5000 janelas, uma a cada 3 dias) em vez das 59 linhas
   de evento — só assim existe "regime de operação" a descobrir;
2. **log1p + `RobustScaler`**, porque a série é fortemente assimétrica e sem isso a silhueta é
   sequestrada por meia dúzia de excursões;
3. **`k` escolhido pela silhueta** (critério interno). O rótulo entra apenas *depois*, para medir
   o **lift** de cada regime — a mesma régua do Baseline;
4. **Decomposição cross-reator**, **idade de campanha** e **qualidade de janela**, que são as
   dimensões que a série univariada sozinha não tem.

## Insumos: um contexto único para features e modelos

`preparar_contexto` monta de uma vez todas as séries auxiliares e devolve um dicionário —
passar um único `contexto` em vez de meia dúzia de mapas evita o erro de calcular uma feature
com um recorte e outra com outro. O que entra nele:

| Item | O que é | Por que existe |
|---|---|---|
| `margem`, `posto` | mediana diária do reator menos a mediana dos três; posição do reator no dia | com três séries a mediana é o valor do meio, então a margem é positiva só para o reator que lidera. Lift 5.3x–6.9x contra 3.6x do nível absoluto |
| `baseline` | quantil 0.99 móvel de 365 d de cada reator | o drift (mediana anual ~2.7 → ~1.8 ppm) faz um limiar fixo significar coisas diferentes em 2013 e 2025 |
| `controle` | EWMA (baseline rolante) e CUSUM dinâmico, amostra a amostra | as cartas viravam gráfico e paravam ali; aqui a estatística no instante da âncora vira **feature** — as duas acumulam desvio pequeno e persistente, que `max` e mediana ignoram |
| `campanhas`, `reparos`, `falhas` | histórico do equipamento pela planilha de inspeção | idade de campanha, reparos desde a última troca e falhas anteriores: informação que a série de ferro não tem como conter |

In [ ]:
CONTEXTO = preparar_contexto(MAP_MEDICOES_BASE, df_inspecoes)

# Atalhos usados nas células seguintes (tudo vem do mesmo contexto, de propósito)
diario_mediana      = CONTEXTO["diario"]
comum_planta        = CONTEXTO["comum"]
margem_cross_reator = CONTEXTO["margem"]
MAP_MARGEM          = {c: CONTEXTO["margem"][c] for c in MAP_MEDICOES_BASE}
CAMPANHAS           = CONTEXTO["campanhas"]

print("\nTrocas de reator conhecidas:", {c: len(v) for c, v in CAMPANHAS.items()})

## Janelas deslizantes

Uma âncora a cada 3 dias, janela de 15 dias, contexto de 90 dias. Janelas sem suporte
(menos de 8 amostras) são **descartadas explicitamente** em vez de virarem estatística de 2 pontos —
esse era um viés silencioso: os eventos variavam de 2 a 150 amostras na janela.

`pre_falha` marca as janelas seguidas de falha em até 15 dias. Ele **não** entra em nenhum ajuste,
só na avaliação.

In [ ]:
df_janelas = gerar_janelas_deslizantes(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA,
    passo_dias=3, janela_dias=15, janela_base_dias=90,
    contexto=CONTEXTO
)

# Mesmas features calculadas exatamente nas 31 âncoras de falha
df_janelas_falha = janelas_nas_falhas(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, contexto=CONTEXTO)

print(f"\nJanelas nas âncoras de falha: {len(df_janelas_falha)}")
print(f"Features por janela: {len([c for c in COLUNAS_MODELO_JANELA if c in df_janelas.columns])}")
df_janelas.head()

## Regimes de operação

`k` escolhido pela silhueta. A tabela do grid mostra também, para cada `k`, o melhor lift
alcançável — inclusive o `lift_util`, restrito a regimes que cobrem pelo menos 20% das falhas.
Sem essa coluna, um cluster com 5 janelas e lift 27x parece um achado; com ela fica claro que ele
cobre 1 falha de 31.

In [ ]:
res_regimes = clusterizar_regimes(df_janelas, df_janelas_falha)

In [ ]:
fig_regimes = plotar_regimes_pca(res_regimes, df_janelas_falha,
                                titulo="Regimes de operação — janelas de 15 dias")
fig_regimes.show()

### Conclusão dos regimes

**Não existe regime pré-falha com assinatura própria** — e na base de 37 falhas a conclusão ficou
ainda mais seca que na de 31:

- o `k` escolhido pela silhueta (k=2) é **degenerado**: 99.6% das janelas em um cluster, com
  lift 0.98 — exatamente a taxa base;
- em todo `k` testado, o cluster de lift alto é minúsculo (0.1% a 0.4% do tempo) e cobre
  **1 falha de 37** (2.7%). Subindo `k` o lift sobe (7.5 → 22.4) e a cobertura não sai do lugar:
  o algoritmo está isolando as excursões extremas, isto é, **redescobrindo o limiar**;
- a coluna `lift_util` (melhor lift entre regimes que cobrem ≥20% das falhas) fica **abaixo de
  1.0 em todos os k** — nenhum regime com cobertura útil supera o acaso. (Na base de 31 havia um
  regime com lift 3.3 em k=4; ele não sobreviveu às falhas resgatadas, que são justamente as de
  pouco sinal.)

Coerente com o Baseline: nos limiares confiáveis o lead time mediano é ~0-3 dias — o sinal é
impulso, não rampa. Um agrupamento de estatísticas da própria série não tem como separar o que a
série não distingue.

**Consequência prática:** clusterização não é caminho para a etapa supervisionada — nem sobre
eventos (removida), nem sobre janelas (mantida aqui como evidência). O ganho tem que vir de
**dimensões novas**, que é o que as três subseções abaixo trazem.

## Decomposição cross-reator: o que é da planta e o que é do reator

A matriz de correlação acima mostra um fato que muda a leitura do problema: **C1 e C2 andam juntos
(0.76), mas o C3 é quase independente (0.11)** — ele é de outro trem. E o componente comum responde
por apenas ~6% da variância: "todos subiram juntos" é a exceção, não a regra.

A pergunta que interessa é se a **margem** (o quanto este reator está acima dos outros) informa mais
que o **nível absoluto**. Comparação na mesma régua — âncoras de falha contra as ~5000 âncoras de base:

In [ ]:
lift_absoluto = avaliar_lift_series(
    {c: diario_mediana[c] for c in MAP_MEDICOES_BASE},
    df_janelas, df_janelas_falha, [3.0, 3.5, 4.0, 5.0], rotulo="mediana diária absoluta"
)
lift_margem = avaliar_lift_series(
    MAP_MARGEM, df_janelas, df_janelas_falha, [0.5, 0.8, 1.2, 1.6, 2.0], rotulo="margem cross-reator"
)

pd.concat([lift_absoluto, lift_margem], ignore_index=True)

### Conclusão do cross-reator

Na mesma régua (37 âncoras de falha contra ~4969 âncoras de base), a margem entrega **lift 3.8x
(>0.8) a 5.8x (>1.6)** contra **3.1x** do melhor critério absoluto (mediana diária > 5 ppm), com
taxa base igual ou menor. Traduzindo: mesma quantidade de alarme, mais detecção — e uma variável
que não envelhece com o drift, porque compara reatores no mesmo dia.

Um cuidado medido e descartado: o filtro **binário** de simultaneidade ("ignorar a ultrapassagem
se outro reator também passou") **piora** o detector — a precisão cai de 0.045 para 0.035. É a
versão **contínua** (o quanto este reator lidera) que carrega informação, não o "está sozinho ou
não".

## Idade de campanha

O tempo desde a última troca do reator é uma variável que o repositório já tem e que ninguém estava
usando. O ponto metodológico: **normalizar pela exposição**. Contar só as falhas sugere "quanto mais
velho, pior"; dividindo pelo tempo que cada reator passou em cada faixa, o risco tem outro formato.

In [ ]:
perfil_hazard_campanha(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, CAMPANHAS)

### Conclusão da campanha

Com as campanhas **corrigidas pela auditoria** (o C2 ganhou as trocas de 2009, 2012 e 2016 que a
regex antiga perdia) e as 37 falhas, o perfil mudou de forma: o risco é aproximadamente estável
nos primeiros 5 anos de campanha (**2.5 a 3.1 falhas por 1000 reator-dias**) e cai ~4x acima de
5 anos (**0.72**). A leitura anterior ("pico em 1-2 anos") era em parte artefato de campanhas mal
contadas no C2.

O que permanece: o perfil **não é monotônico** — "quanto mais velho, pior" continua falso, e a
variável segue entrando no modelo como **faixa ordinal** (`faixa_campanha`), não como número de
dias em regressão linear.

Ressalvas: são poucas trocas conhecidas por reator, a relação entre os apontamentos de troca de
2016 do C2 (fevereiro e julho) precisa de confirmação da planta (está na ficha de validação), e
campanhas longas sobreviventes carregam viés de sobrevivência.

## O confundidor que atravessa tudo: as falhas estão no passado

Antes de olhar qualquer feature, é preciso olhar **quando** as falhas acontecem. Se elas se
concentram em um período, qualquer variável que também mude com o tempo vira preditora
espúria — e a validação temporal fica sem poder, porque testar no futuro significa testar
com pouquíssimos positivos.

In [ ]:
perfil_temporal_falhas(MAP_EVENTOS_FALHA, MAP_MEDICOES_BASE)

print()
df_tendencia = testar_tendencia(MAP_MEDICOES_BASE)

### O que essas duas tabelas impõem ao resto do estudo

**31 das 37 falhas (84%) acontecem até 2018.** A taxa cai de ~4.3 falhas por 1000 reator-dias em
2016-2018 para **0.30** em 2019-2021, com retomada parcial em 2022-2024 (**1.52** — os três
vazamentos em plug de reparo do C3 resgatados na auditoria de eventos). Ao mesmo tempo, a série
tem tendência de queda confirmada (tau de Kendall ≈ **-0.29** com p astronômico nos três reatores;
inclinação de Sen ≈ **-0.05 ppm por ano**; mediana de 2.45-2.50 ppm em 2011-2015 contra 1.80-1.90
em 2021-2026).

Ou seja: **ferro alto e falha caem juntos ao longo do tempo, sem que um cause o outro.**
Qualquer feature de nível absoluto vai parecer preditiva só por ser mais alta no passado.
É por isso que a tabela seguinte reporta a **AUC ajustada por época** ao lado da AUC bruta — e
é por isso que a série não estacionária justifica limite móvel e features relativas.

## Valor individual de cada feature

AUC de Mann-Whitney de cada feature contra `pre_falha`, medida nas ~4969 janelas (e não nos
~60 eventos, onde não há amostra para a medida significar algo). `AUC_ajustada` recalcula a
separação **dentro de blocos de 3 anos** e faz a média ponderada pelos positivos: quando a AUC
bruta se afasta de 0.5 e a ajustada volta para perto, a feature era um relógio disfarçado.

In [ ]:
df_valor_features = avaliar_valor_features(df_janelas)
df_valor_features

### Conclusão — e ela inverte o senso comum do estudo

(Números da base de 37 falhas / 177 janelas pré-falha, 11/08/2026.)

| Feature | AUC bruta | AUC ajustada | Leitura |
|---|---|---|---|
| `mediana` | 0.627 | **0.507** | a "melhor feature" era quase toda efeito de época |
| `p90` | 0.606 | **0.517** | idem |
| `razao_max` | 0.375 | **0.451** | a inversão aparente era o drift, não sinal |
| `densidade_relativa` | 0.391 | **0.414** | estável: a amostragem rareia antes da parada |
| `assimetria` / `curtose` | 0.382 / 0.418 | **0.417 / 0.417** | estáveis: a janela pré-falha tem forma diferente (menos assimétrica) |
| `dias_desde_ultimo_reparo` | 0.521 | **0.567** | inverte e vira a mais forte: reparo recente, mais risco |
| `cusum_rel` | 0.552 | **0.556** | estável — a carta de controle como feature carrega sinal próprio |
| `margem_media` | 0.546 | **0.544** | estável |
| `idade_campanha` | 0.430 | **0.536** | ainda inverte com o ajuste, mais fraca que na base anterior |
| `n_reparos_vidro_campanha` | 0.449 | **0.532** | mais reparos de vidro na campanha, mais risco |

Três consequências práticas:

1. **Nível absoluto de ferro é o pior tipo de feature aqui** — não porque não separe, mas
   porque o que ele separa é o ano. Confirma, por outro caminho, o diagnóstico do deck da
   Bayer, e explica por que um limiar fixo envelhece mal.
2. **O que sobrevive ao ajuste é o que não é nível**: histórico de reparo (dias desde o último
   reparo, reparos de vidro na campanha, idade de campanha), forma da janela (assimetria,
   curtose, densidade de amostragem), acúmulo do CUSUM e margem cross-reator. As de histórico
   descrevem **degradação do equipamento**, que é a física do problema.
3. **Nenhuma feature isolada passa de AUC ajustada 0.57.** Não existe variável salvadora neste
   conjunto; o ganho, se houver, vem de combinação — e precisa ser demonstrado com validação
   temporal, não com um split único.

Ressalva sobre a margem: `margem_max` tem AUC ajustada de apenas 0.515, mas lift de **5.8x** no
limiar de 1.6. Não é contradição — AUC é uma medida de separação **média**, lift é medida de
**cauda**. A margem não distingue o dia comum; distingue o dia extremo, que é o que interessa
para alarme.

**Features consideradas e recusadas de propósito:** ano ou indicador de época (seria a mais forte
in-sample e é puro vazamento do confundidor temporal); nível bruto dos outros reatores como
feature (a margem já carrega a parte relativa sem trazer o efeito de planta); interações
explícitas margem × nível (árvores capturam sozinhas). E as que não dá para construir com o que
existe no repositório: variáveis de processo (temperatura, pressão, concentração de soda),
histórico de condenação de batelada e medição de espessura do revestimento — essas três são o
que realmente falta para a etapa supervisionada ter chance.

## A nova régua: regra composta

Juntando o que a etapa não supervisionada encontrou, a regra a bater deixa de ser o patamar fixo:

**`max diário > 20 ppm`  OU  (`mediana diária > 3.0` E `margem cross-reator > 0.6`)**

Os dois ramos existem porque o sinal aparece de duas formas e nenhuma regra única pega as duas:

- **impulso** — uma leitura altíssima isolada (C3 23/11/2013, 264 ppm) não move a mediana diária;
  só o ramo do `max` pega;
- **elevação sustentada** — vários dias em patamar acima do normal, que o `max` de um dia não
  distingue de um pico de laboratório; o ramo da mediana pega, e a **margem** é o que separa
  "este reator subiu" de "a planta subiu".

Sobre as 37 falhas ancoradas: **VP 12, FP 69, F2 0.262**, contra **VP 8, FP 171, F2 0.122** da
regra vigente — mais detecção com 60% menos alarme falso.

**Ressalva obrigatória:** os limiares foram escolhidos olhando as mesmas falhas. Isto é **ponto
de partida para a etapa supervisionada validar no split temporal**, não resultado validado.

In [ ]:
df_detectores = comparar_detectores(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA,
                                   margem=margem_cross_reator)
df_detectores

### O que segue para a etapa supervisionada

| Achado | Como entra na Etapa 3 |
|---|---|
| Margem cross-reator (lift 3.8-5.8x vs 3.1x do absoluto) | features `margem_*`, `posto_medio`, `frac_lider` |
| Limite móvel de 365 d (imune ao drift) | feature `razao_limite_movel` |
| Histórico de reparo e campanha (AUC ajustada 0.53-0.57) | `dias_desde_ultimo_reparo`, `n_reparos_vidro_campanha`, `idade_campanha`, `faixa_campanha` |
| Forma e qualidade da janela | `assimetria`, `curtose`, `densidade_relativa`, `n_amostras_*`, `maior_lacuna_*` |
| Cartas de controle como feature | `ewma_z`, `cusum_rel` |
| Regra composta (F2 0.262 contra 0.122 da vigente) | linha de base a superar, no lugar dos 5 ppm |
| Clusterização de janelas | **descartada** — não separa nada além do limiar |

# Classificação (Abordagem Supervisionada)

In [ ]:
JANELAS       = [15, 12, 9, 6, 3]
N_SPLITS      = 5
RANDOM_STATE  = 42
TEST_SIZE     = 0.2

# Sobrescreve o STATS_FUNCS padrão do utils.py para as células desta seção
STATS_FUNCS = {
    'media':   np.mean,
    'mediana': np.median,
    'std':     np.std,
    'max':     np.max,
    'p75':     lambda x: np.percentile(x, 75),
    'p90':     lambda x: np.percentile(x, 90),
    'range':   lambda x: np.max(x) - np.min(x),
}

# Pista A (ver "Política de tratamento de dados"): treino e avaliação SEMPRE sobre o
# Original (bruto + limpar_excursoes). Os tratamentos deletam as leituras-gatilho e
# invalidam a comparação com a regra vigente (no IQR nada passa de 4.0 ppm).
MAP_MEDICOES = {
    "Original": {"C1": df_crystallizer1, "C2": df_crystallizer2, "C3": df_crystallizer3},
}

CONFIGS = {
    "C1": {"Original": (df_crystallizer1, df_eventos_crystallizer1)},
    "C2": {"Original": (df_crystallizer2, df_eventos_crystallizer2)},
    "C3": {"Original": (df_crystallizer3, df_eventos_crystallizer3)},
}

# As features de contexto (margem cross-reator, limite móvel, EWMA/CUSUM, idade de campanha,
# suporte da janela) entram por CONTEXTO, montado na seção não supervisionada. Rodando com
# `contexto=None` o conjunto volta a ser o antigo — é assim que o ganho fica auditável.
#
# ATENÇÃO: os relatórios abaixo usam o split temporal único do `dividir_dados`, que corta
# pelo quantil dos POSITIVOS e INVERTE a prevalência entre treino e teste (no unificado:
# 80% de positivos no treino contra 25% no teste). Um modelo treinado assim prevê positivo
# quase sempre — daí o recall 1.00 com precisão igual à taxa base. Eles ficam aqui como
# histórico; a conclusão sobre modelo está na seção "Validação", com CV temporal.

## Escopo do ajuste: unificado, e por quê

**Aqui existiam três blocos de classificação — um por reator — que foram removidos.** A decisão
de escopo do estudo é: **ajuste unificado, alarme e prestação de contas por reator** (a tabela
completa está em "Resultados por reator", no fim do notebook). O motivo de o ajuste ser unificado
(base de 37 falhas):

| Reator | Falhas | Janelas pré-falha antes do 1º corte temporal |
|---|---|---|
| C1 | 12 | 43 |
| C2 | 16 | 55 |
| C3 | **9** | **15** |
| unificado | 37 | **113** |

Um modelo ajustado só no C3 teria 15 janelas positivas para aprender. A célula abaixo mede isso
fora da amostra em vez de argumentar.

In [ ]:
# Treinar unificado, só no próprio reator, ou nos outros dois?
# Métrica: PR-AUC dividida pela taxa base do próprio reator (as taxas base diferem entre eles).
df_escopo_treino = comparar_escopo_treino(df_janelas)

### Conclusão do escopo

Nenhum escopo vence nos três reatores, e **"próprio" não vence em nenhum**: no C1 e no C2 o
unificado é o melhor (lift 3.79 e 1.32), e no C3 o melhor é treinar **nos outros dois**
(lift 3.47 contra 1.15 treinando em si mesmo) — com 9 falhas, das quais 3 sem sinal de ferro,
o próprio reator não tem material para ensinar nada.

Some-se que a melhor feature do estudo — a **margem cross-reator** — só existe com as três séries
juntas: separar completamente não é sequer possível.

**Portanto: o ajuste (features e modelos) roda unificado.** O que passa a ser por reator é o
**limiar de alarme** e o **relatório**, nas seções seguintes.

## Ajuste unificado (canônico)

Este é o único ajuste supervisionado do estudo. Ele usa as janelas de cada evento recortadas na
série do **próprio** reator (`map_medicoes_unif`) e as features de contexto (`CONTEXTO`) — ou seja,
unifica a amostra sem misturar a medição de reatores diferentes dentro de uma janela.

In [ ]:
MODO = "Unificado"

# Pista A: só o Original entra nos modelos (ver "Política de tratamento de dados")
CONFIGS_UNIFICADO = {
    "Original": df_crystallizer123,
}

# map_medicoes_unif corrige um defeito que existia aqui: sem ele, a janela de cada evento era
# recortada sobre o dataframe concatenado dos três reatores (182 amostras em vez das 65 do
# reator do evento), então max/p90/range vinham de outro equipamento.
resultados_por_tratamento_unif, df_cv_consolidado_unif = rodar_classificacao_por_tratamento(
    MODO, CONFIGS_UNIFICADO, df_eventos_crystallizer123, JANELAS,
    stats_funcs=STATS_FUNCS, test_size=TEST_SIZE, usar_smote=False,
    map_medicoes_unif=MAP_MEDICOES["Original"], contexto=CONTEXTO
)

# Validação: protocolo, fora da amostra e custo

Esta seção existe porque o relatório supervisionado acima **não sustenta conclusão**. O corte
do `dividir_dados` usa o quantil dos positivos e a prevalência inverte entre treino e teste
(unificado: 24 positivos / 6 negativos no treino, 7 / 21 no teste). Um modelo treinado com 80%
de positivos prevê positivo quase sempre — e é o que se vê: **recall 1.00 em 11 das 12
combinações**, com precisão igual à taxa base do bloco de teste. Um dos blocos do C3 chega a
F1 = 1.00 com cinco linhas.

O que esta seção faz, em ordem:

1. **CV temporal de janela expansiva**, com a prevalência de cada fold visível e F2 (a métrica
   alvo) no lugar de F1 — nas duas tabelas de evento, LC 10 e LC 5;
2. **modelo avaliado como detector**, que é a única comparação justa com a regra vigente:
   pontuar janelas, agrupar alarmes em 15 dias, contar VP/FP contra as falhas do mesmo período;
3. **validação da regra composta fora da amostra** — holdout temporal, leave-one-reactor-out e
   bootstrap;
4. **lead time** evento a evento, o número que decide o projeto;
5. **ponto de operação por custo**, para quando a planta trouxer o custo do alarme falso e o
   da falha perdida.

## 1. CV temporal nas duas tabelas de evento

Treina no passado, testa no bloco seguinte, repete em quatro cortes. Modelos fixos e
balanceados por classe, sem grid interno: com ~30 eventos por fold, um GridSearchCV dentro do
fold escolhe hiperparâmetro por ruído.

In [ ]:
# LC 10 ppm — negativos "difíceis" (excursão clara que não terminou em falha)
feat_lc10, cols_lc10 = extrair_features(
    "Unificado", df_crystallizer123, df_eventos_crystallizer123, JANELAS, STATS_FUNCS,
    map_medicoes_unif=MAP_MEDICOES["Original"], contexto=CONTEXTO)

print("=" * 70); print("LC 10 ppm"); print("=" * 70)
df_cv_lc10, resumo_cv_lc10 = avaliar_cv_temporal(feat_lc10, cols_lc10)

In [ ]:
# LC 5 ppm — o limiar em que a planta realmente opera, com ~3x mais negativos
feat_lc5, cols_lc5 = extrair_features(
    "Unificado", df_crystallizer123, df_eventos_crystallizer123_lc5, JANELAS, STATS_FUNCS,
    map_medicoes_unif=MAP_MEDICOES["Original"], contexto=CONTEXTO)

print("=" * 70); print("LC 5 ppm (operacional)"); print("=" * 70)
df_cv_lc5, resumo_cv_lc5 = avaliar_cv_temporal(feat_lc5, cols_lc5)

print("\n" + "=" * 70)
print("Comparação LC 10 vs LC 5 (F2 médio entre folds)")
print(pd.concat([resumo_cv_lc10["F2_medio"].rename("LC 10"),
                 resumo_cv_lc5["F2_medio"].rename("LC 5")], axis=1).to_string())

### Leitura da CV temporal

O padrão central continua o da prevalência: o corte temporal herda o desequilíbrio da base
(70-82% de positivos no treino contra 22-44% nos blocos de teste), e o fold 1 — cujo bloco de
teste tem 90% de positivos — entrega F2 ~0.98 para qualquer modelo, o que é artefato, não mérito.

Com a base de 37 falhas os folds tardios melhoraram (as falhas resgatadas de 2023-2024 dão
positivos ao teste), mas os blocos têm **9 a 10 linhas** — um acerto a mais ou a menos move o F2
em 0.2-0.4, e o desvio entre folds (0.2-0.5) segue da ordem da diferença entre modelos.
**Ranking de modelo neste conjunto continua sendo ruído**; a comparação que decide é a do
detector, abaixo.

## 2. Modelo avaliado como detector

Aqui o modelo é treinado sobre as ~5000 janelas deslizantes (rótulo `pre_falha`, taxa base 3%,
sem classe negativa sintetizada por limiar), pontua o bloco seguinte, e as janelas acima do
quantil viram alarme agrupado em 15 dias. A regra composta e a regra vigente são medidas **no
mesmo período de teste** — sem isso a comparação não vale.

In [ ]:
res_detector = treinar_detector_janelas(
    df_janelas, MAP_EVENTOS_FALHA,
    margem=margem_cross_reator, map_medicoes=MAP_MEDICOES_BASE
)

### O resultado que decide o rumo da Etapa 3

Fora da amostra, no mesmo período de teste (3 folds, 1988 janelas pontuadas, 13 falhas):

| Regra | VP | FP | precisão | recall | F2 |
|---|---|---|---|---|---|
| **regra composta (20 / 3.5 / 0.9)** | 3 | 19 | 0.136 | 0.231 | **0.203** |
| regra composta (20 / 3.0 / 0.6) | 3 | 28 | 0.097 | 0.231 | 0.181 |
| modelo (quantil 0.90) | 3 | 48 | 0.059 | 0.231 | 0.146 |
| regra vigente (`max > 5`) | 2 | 64 | 0.030 | 0.154 | 0.085 |
| modelo (quantis 0.95 / 0.98 / 0.99) | 0 | 5 / 0 / 0 | 0.000 | 0.000 | 0.000 |

**A regra composta bate o modelo supervisionado e mais que dobra o F2 da regra vigente.** O
modelo empata em recall com a composta no quantil mais permissivo, mas ao custo de 2.5x mais
alarme falso — e some nos quantis seletivos: ele não consegue pôr as janelas de pré-falha no topo
do ranking.

Não é um resultado contra machine learning em geral: é o esperado com 37 positivos, 84% deles
concentrados até 2018, e uma única variável medida. O modelo tem graus de liberdade demais para o
que a base sustenta.

**Consequência para a Etapa 3: o entregável é a regra, não o modelo.**

## 3. A regra composta fora da amostra

Os limiares foram escolhidos olhando as mesmas 31 falhas. Três validações independentes:
holdout temporal (ajusta no passado, mede no futuro), leave-one-reactor-out (ajusta em dois
reatores, mede no terceiro) e bootstrap das falhas (intervalo do F2).

In [ ]:
validacao_regra = validar_regra_composta(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, margem=margem_cross_reator,
    data_corte="2019-01-01", limiares_fixos=(20, 3.0, 0.6), n_bootstrap=500
)

### Leitura das três validações — a mais importante desta seção

**Holdout temporal: o resultado endureceu, e isso é informação.** Depois de 01/01/2019 agora
existem **6 falhas** (o resgate dobrou de 3 para 6) — e **nenhuma regra acerta nenhuma delas**,
incluindo a vigente (que ainda gera 86 alarmes falsos no período). As falhas pós-2019 são
majoritariamente vazamentos em plug de reparo e trocas constatadas em inspeção: **modos que
liberam pouco ferro**. Isso não é mais só "teste sem poder" — é um indício de que o regime
recente de falhas é diferente do que o ferro consegue ver, e precisa constar de qualquer entrega.

**Leave-one-reactor-out: o sinal continua concentrado no C3, atenuado.**

| Reator avaliado | VP | FP | recall | F2 |
|---|---|---|---|---|
| C3 | 5 de 9 | 29 | **0.556** | **0.357** |
| C1 | 2 de 12 | 11 | 0.167 | 0.164 |
| C2 | 1 de 16 | 16 | 0.062 | 0.062 |

Com limiares ajustados **nos outros dois reatores**, a regra encontra 5 das 9 falhas do C3 —
as 5 antigas; os 3 vazamentos em plug resgatados não mostram ferro. No C1 e no C2, quase nada.

**Bootstrap:** F2 = 0.256, IC95% **[0.156, 0.363]**. O limite inferior segue acima do F2 da
regra vigente (0.122), mas o intervalo é largo — como esperado com 37 positivos.

## 4. Lead time — o número que decide o projeto

In [ ]:
df_lead = lead_time_regra(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, margem=margem_cross_reator,
    limiares=(20, 3.0, 0.6), dias_max=60
)

df_lead[df_lead["alarmou"]].sort_values("lead_dias", ascending=False)

### Existe antecedência — para metade das falhas alarmadas

A regra composta alarma em **17 das 37 falhas** dentro de 60 dias, com lead mediano de
**3.0 dias** (p25 1.0, p75 23.0, máximo 40). Treze falhas têm lead ≥ 1 dia e **8 têm lead ≥
7 dias**.

Muda a conversa com a planta: para cerca de um quinto das falhas há **uma semana ou mais** de
antecedência; para as demais, o alarme chega junto com o evento — e para as falhas de pouco
ferro (plugs, agitador) não chega. O produto honesto é **"antecipação parcial + confirmação com
menos alarme falso"**, não "predição de falha".

## 5. Ponto de operação por custo

Enquanto o histórico de condenação indevida e o custo de parada não planejada não chegam da
planta, o ponto de operação é escolhido pelo F2 — que embute uma razão arbitrária (recall vale
4x a precisão). Com os dois custos, a escolha vira aritmética. A tabela abaixo mostra qual
regra vence para cada razão custo(falha perdida)/custo(alarme falso).

In [ ]:
print(sensibilidade_custo(df_detectores, n_falhas=31).to_string(index=False))

print("\nCusto esperado com razão 25:1 (uma falha perdida vale 25 alarmes falsos):")
custo_esperado(df_detectores, custo_fp=1, custo_fn=25, n_falhas=31).head(6)

### Fecho da validação

| Pergunta | Resposta medida (base de 37 falhas) |
|---|---|
| Os modelos supervisionados superam a regra? | **Não.** Fora da amostra, F2 0.146 contra 0.203 da regra composta |
| A regra composta supera a regra vigente? | **Sim, dentro da amostra** (0.262 vs 0.122) e no período de teste do detector (0.203 vs 0.085); IC95% do bootstrap [0.156, 0.363] |
| A regra se valida no futuro? | **Não nas falhas recentes** — 6 falhas pós-2019, 0 detectadas: os modos recentes (plug, constatada em inspeção) liberam pouco ferro |
| A regra vale para os três reatores? | **Não** — recall 0.556 no C3, 0.167 no C1, 0.062 no C2 (LORO) |
| Existe antecedência? | **Parcial** — 17 de 37 alarmadas, lead mediano 3 dias, 8 falhas com ≥ 7 dias |
| Qual ponto de operação? | Depende da razão de custo: até ~5:1 vence `max > 20`; de 10:1 em diante vence a regra composta |

**Recomendação de entrega:** a regra composta como sistema de alarme, com escopo declarado
(desempenho concentrado no C3 e nos modos de falha com liberação de ferro), acompanhada da ficha
de eventos para a planta validar as datas e da classificação de modo de falha como pedido
prioritário. Modelo supervisionado só volta à mesa com variáveis de processo além do ferro ou
com mais eventos rotulados.

# IsolationForest sobre janelas deslizantes

## O que foi removido, e por quê

A versão anterior treinava o IsolationForest sobre as **59 linhas da tabela de eventos**
(`JANELAS = [6, 3, 1]`, 4 estatísticas, grid de 108 combinações). Problemas:

1. 59 linhas não sustentam um grid de 108 combinações — o "melhor F2 de treino" que escolhia o
   modelo é ruído de seleção, e a matriz de confusão saía de 6 a 12 linhas de teste;
2. detectar anomalia **sobre a tabela de eventos é circular**: a tabela foi construída selecionando
   ultrapassagens, então "anômalo" e "evento" eram quase sinônimos por construção;
3. não havia comparação com taxa base — não dava para saber se o alarme significava alguma coisa.

Agora ele roda onde faz sentido: sobre **toda a operação** (as janelas deslizantes), sem rótulo no
ajuste, avaliado por **lift** e pela mesma convenção de alarme dos demais detectores.

In [ ]:
res_iforest = rodar_iforest_janelas(
    df_janelas, df_janelas_falha, map_falhas=MAP_EVENTOS_FALHA,
    contaminacao_grid=(0.01, 0.02, 0.05, 0.10)
)

In [ ]:
fig_iforest = plotar_iforest_janelas(res_iforest, contaminacao=0.05)
fig_iforest.show()

### Conclusão do IsolationForest

O lift é real (3x a 9x conforme a contaminação), mas o **F2 fica em 0.03–0.08**, uma ordem de
grandeza abaixo da regra composta (0.268) e da própria regra vigente (0.132): ele marca 5% do tempo
para capturar 8 das 31 falhas, gerando 79 alarmes falsos.

Diagnóstico: o IsolationForest mede **distância da operação normal em todas as direções**, e a
maior parte dessa distância é composta por coisas que nada têm a ver com falha — lacuna de
amostragem, mudança de regime da série, janela com poucas amostras. Ou seja, ele reencontra o mesmo
teto da clusterização.

**Decisão: não seguir por detecção de anomalia genérica.** O caminho que carrega sinal é o
direcional — margem cross-reator e nível relativo ao próprio baseline —, já incorporado às features
da etapa supervisionada.

# Cartas de controle

Avaliando EWMA e CUSUM: as cartas disparam com antecedência em relação à falha?

Duas mudanças em relação à versão anterior desta seção:

1. o EWMA passa a usar **baseline rolante** em vez do baseline escolhido à mão em 2012-2015 —
   a série não é estacionária (tau de Kendall -0.29), então um baseline fixo envelhece;
2. as duas cartas deixaram de ser só gráfico: `ewma_z` e `cusum_rel` no instante da âncora
   viraram **features** dos modelos (ver "Insumos" na seção não supervisionada). O CUSUM é uma
   das poucas features que sobrevivem ao ajuste por época (AUC ajustada 0.567).

## EWMA

In [ ]:
# EWMA com BASELINE ROLANTE — substitui o baseline escolhido à mão em 2012-2015.
#
# Por quê: a série tem tendência de queda confirmada (tau de Kendall -0.29, p astronômico;
# inclinação de Sen ~ -0.05 ppm/ano; mediana 2.45 -> 1.85 ppm entre 2011-2015 e 2021-2026).
# Um baseline fixo naquele período deixa o limite sistematicamente frouxo no regime atual, e o
# grid antigo compensava isso levando L até 13 — ou seja, ajustando ruído. No baseline rolante a
# média e a dispersão vêm de uma janela móvel do passado (shift(1) impede que o ponto de hoje
# entre no próprio limite) e a dispersão é robusta (IQR/1.349), porque a série tem excursões de
# três ordens de grandeza.
#
# Leitura esperada da comparação abaixo: com o mesmo L, o baseline rolante dispara MAIS que o
# fixo — a dispersão do regime atual é bem menor que a do baseline de 2012-2015, então o limite
# acompanha a série em vez de ficar alto e imóvel. Esse é o comportamento correto para uma série
# não estacionária, mas significa que a carta precisa de L maior para servir como alarme isolado.
# O valor principal do EWMA rolante aqui não é a carta: é a feature "ewma_z" (e a "cusum_rel"),
# que entram nos modelos — o CUSUM é uma das poucas que sobrevivem ao ajuste por época
# (AUC ajustada 0.567, contra 0.524 da mediana).
serie_fe = (df_crystallizer1.set_index("TIMESTAMP")["Resultado de Ferro (ppm)"]
            .dropna().sort_index())
df_eventos_c1 = df_eventos_crystallizer1.copy()

# --- versão antiga: baseline fixo escolhido à mão
periodos_baseline = [("2012-03-01", "2013-03-25"), ("2014-01-01", "2015-01-01")]
serie_baseline = pd.concat([serie_fe.loc[i:f] for i, f in periodos_baseline])
media_historica, std_historico = serie_baseline.mean(), serie_baseline.std()
print(f"Baseline fixo | média {media_historica:.3f} ppm | std {std_historico:.3f} ppm "
      f"| {len(serie_baseline)} amostras")

df_ewma_fixo = calcular_ewma(serie_fe, media_historica, std_historico, lambd=0.2, L=7)
_, _, f2_fixo = avaliar_carta_controle(df_ewma_fixo, df_eventos_c1,
                                       "EWMA baseline fixo (2012-2015), L=7", janela_dias=15)

# --- versão nova: baseline rolante
df_ewma = calcular_ewma_rolante(serie_fe, lambd=0.2, L=7, janela_baseline=500)
_, _, f2_rolante = avaliar_carta_controle(df_ewma, df_eventos_c1,
                                          "EWMA baseline rolante, L=7", janela_dias=15)

print(f"\nAlarmes: baseline fixo {int(df_ewma_fixo['Alarme'].sum())} "
      f"| baseline rolante {int(df_ewma['Alarme'].sum())}")
print(f"F2: fixo {f2_fixo:.4f} | rolante {f2_rolante:.4f}")

fig_ewma = plotar_carta_ewma(df_ewma, df_eventos_c1,
                             titulo="Carta EWMA com baseline rolante — Crystallizer #1")
fig_ewma.show()

## CUMSUM

In [ ]:
# ==========================================
# 2. FUNÇÃO DE PLOTAGEM
# ==========================================

# ==========================================
# 3. BLOCO DE EXECUÇÃO E OTIMIZAÇÃO
# ==========================================

# Preparação dos dados
df_c1 = df_crystallizer1.copy()
df_c1['TIMESTAMP'] = pd.to_datetime(df_c1['TIMESTAMP'])
df_c1 = df_c1.sort_values('TIMESTAMP').set_index('TIMESTAMP')
serie_fe = df_c1['Resultado de Ferro (ppm)'].dropna()

df_eventos_c1 = df_eventos_crystallizer1.copy()
df_eventos_c1['TIMESTAMP'] = pd.to_datetime(df_eventos_c1['TIMESTAMP'])

# Definição do Grid de Parâmetros para buscar a melhor performance
param_grid_cusum = {
    'janela_baseline': [15, 30, 45, 60], # Quantos dias passados compõem o "normal"
    'k': [0.25, 0.5, 0.75],              # Folga (menor = soma desvios menores)
    'h': [3, 4, 5, 6],                   # Limite de alarme (maior = mais rigoroso)
    'janela_dias': [3, 6, 9, 12]         # Janela de antecedência para prever a falha
}

# Roda a otimização
melhores_parametros, max_f2 = otimizar_cusum_dinamico(serie_fe, df_eventos_c1, param_grid_cusum)

# Gera a carta final com os hiperparâmetros campeões
df_cusum_otimizado = calcular_cusum_dinamico(
    serie_fe, 
    janela_baseline=melhores_parametros['janela_baseline'], 
    k=melhores_parametros['k'], 
    h=melhores_parametros['h']
)

# Avalia formalmente para exibir o relatório
nome_modelo = f"CUSUM Dinâmico (jan_base={melhores_parametros['janela_baseline']}, k={melhores_parametros['k']}, h={melhores_parametros['h']})"
avaliar_carta_controle(df_cusum_otimizado, df_eventos_c1, nome_carta=nome_modelo, janela_dias=melhores_parametros['janela_dias'])

# Plota o gráfico para análise visual (a função devolve a fig; quem chama dá o .show())
fig_cusum = plotar_carta_cusum(
    df_cusum_otimizado, df_eventos_c1,
    titulo=f"Carta CUSUM Dinâmico - C1 {nome_modelo}",
    mostrar_falsos=False
)
fig_cusum.show()

# Resultados por reator

Até aqui todo resultado foi reportado somando os três reatores. Esta seção separa — e a
separação muda a leitura do projeto.

**A regra de trabalho do estudo, medida e não arbitrada (base de 37 falhas):**

| Etapa | Decisão | Evidência |
|---|---|---|
| Carga, limpeza e EDA distribucional | **unificar** | distribuições estatisticamente idênticas: Kruskal-Wallis η² = 0.0008; mediana 2.2 / 2.3 / 2.3 ppm; p99 = 4.3 nos três |
| Construção de features | **unificar (obrigatório)** | margem e posto cross-reator só existem com as três séries |
| Ajuste (modelo e limiar) | **unificar** | "próprio" não vence em nenhum reator; o C3 aprende melhor com os outros dois (lift 3.47) do que consigo (1.15) |
| Calibração do alarme | **por reator** | ver a tabela de calibração abaixo |
| Avaliação e relatório | **por reator, sempre** | o número agregado esconde que a regra funciona no C3 e falha no C2 |

## A qual estatística cada reator responde

Antes de calibrar, vale ver *o que* dispara em cada equipamento. Os três têm a mesma distribuição
de ferro, mas não a mesma relação entre ferro e falha.

In [ ]:
linhas_lift = []
for _cryst in MAP_MEDICOES_BASE:
    _J = df_janelas[df_janelas["Crystallizer"] == _cryst]
    _JF = janelas_nas_falhas({_cryst: MAP_MEDICOES_BASE[_cryst]},
                             {_cryst: MAP_EVENTOS_FALHA[_cryst]}, contexto=CONTEXTO)
    _abs = avaliar_lift_series({_cryst: diario_mediana[_cryst]}, _J, _JF, [3.0, 3.5],
                               rotulo=f"{_cryst} mediana diária")
    _mar = avaliar_lift_series({_cryst: MAP_MARGEM[_cryst]}, _J, _JF, [0.6, 1.2],
                               rotulo=f"{_cryst} margem")
    linhas_lift.append(pd.concat([_abs, _mar], ignore_index=True))

df_lift_por_reator = pd.concat(linhas_lift, ignore_index=True)
df_lift_por_reator

### Cada reator tem um gatilho diferente

| Critério (lift) | C1 | C2 | C3 |
|---|---|---|---|
| mediana diária > 3.5 | 1.30 (2 de 12) | 2.26 (4 de 16) | **4.11 (5 de 9)** |
| margem > 0.6 | **4.32 (4 de 12)** | 2.17 (3 de 16) | **4.58 (6 de 9)** |
| margem > 1.2 | **11.22 (2 de 12)** | 2.26 (1 de 16) | 4.42 (2 de 9) |

**C3 responde a nível e a margem, C1 responde só à margem, C2 responde pouco a qualquer
estatística.** É por isso que um limiar único é subótimo: ele precisa servir a três
comportamentos diferentes ao mesmo tempo.

## Calibração do alarme por reator

A **estrutura** da regra continua a mesma nos três (`max > A` OU (`mediana > B` E `margem > C`)) —
o que muda é o ponto de corte. Isso é importante para a implantação: é uma regra só, com uma
tabela de três linhas de parâmetro, e não três lógicas diferentes para a operação decorar.

**Ressalva obrigatória:** com 6 a 14 falhas por reator, o limiar próprio é ajustado *in-sample* e
é um teto otimista. Deve ser lido como "quanto se ganharia se o limiar fosse do equipamento", e
revisto quando a planta trouxer mais eventos rotulados.

In [ ]:
LIMIARES_POR_REATOR, df_calibracao, LIMIAR_GLOBAL = calibrar_limiares_por_reator(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, margem=margem_cross_reator
)

In [ ]:
df_regra_calibrada, ALARMES_CALIBRADOS = avaliar_regra_calibrada(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, LIMIARES_POR_REATOR,
    margem=margem_cross_reator, limiares_globais=(20, 3.0, 0.6)
)

### O ganho da calibração — e por que ele não sobrevive à validação

Dentro da amostra, calibrar por reator parece claramente melhor:

| Regra | VP | FP | precisão | recall | F2 |
|---|---|---|---|---|---|
| composta com limiar por reator | 14 | 59 | 0.192 | 0.389 | **0.323** |
| composta com limiar único (20/3.0/0.6) | 12 | 69 | 0.148 | 0.333 | 0.267 |
| regra vigente (`max > 5 ppm`) | 8 | 171 | 0.045 | 0.222 | 0.124 |

**Só que esse F2 de 0.323 é escolhido olhando as mesmas falhas que ele detecta.** A seção
seguinte submete a calibração a uma validação honesta (calibra no passado, mede no futuro) e o
resultado se inverte: o limiar único fixo entrega **mais** que o calibrado. Leia a próxima seção
antes de usar a tabela acima.

## Os limiares já são ótimos? Busca ampla, platô e validação

A calibração acima usou uma grade de 4×4×4 = 64 combinações — e **dois dos três reatores
escolheram o valor da borda** (`limite_max = 30`, o teto da grade). Quando o ótimo encosta na
borda, ele não é ótimo: é o limite da busca. Esta seção responde a três perguntas:

1. **os valores em uso são realmente ótimos?** — busca em grade ampla (11×10×10 = 1100
   combinações por reator), incluindo o valor `inf`, que **desliga** o ramo correspondente e
   deixa a própria busca descobrir se um ramo da regra é peso morto;
2. **o ótimo é um pico ou um platô?** — pico isolado é assinatura de sobreajuste; o que se
   implanta é o topo do platô, escolhido pela melhor média na vizinhança da grade;
3. **calibrar compensa?** — validação *walk-forward*: escolhe o limiar usando só o passado de
   cada corte e mede no bloco seguinte, nunca visto.

In [ ]:
resultado_otimizacao = otimizar_regra_por_reator(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, margem=margem_cross_reator
)

In [ ]:
# Os limiares em uso contra o melhor da grade ampla, reator a reator
LIMIARES_EM_USO = {"C1": (30, 2.5, 0.6), "C2": (20, 3.0, 0.6), "C3": (30, 3.5, 0.9)}

df_comparacao_limiares = comparar_limiares_por_reator(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA,
    conjuntos={
        "em uso": LIMIARES_EM_USO,
        "grade ampla (estável)": limiares_da_otimizacao(resultado_otimizacao, "estavel"),
        "grade ampla (argmax)": limiares_da_otimizacao(resultado_otimizacao, "argmax"),
    },
    margem=margem_cross_reator,
)

### Resposta 1: sim, os valores em uso já são ótimos dentro do ruído

Ampliando a busca de 64 para **1100 combinações por reator**, o ganho é de **+0.005 (C1)** e
**+0.021 (C2)**, e no **C3 o valor em uso já é o próprio argmax** (a escolha estável fica 0.015
*abaixo* dele). Para dimensionar: entre as 27 combinações vizinhas do ponto em uso, o F2 varia de
0.21 a 0.28 — ou seja, **os ganhos encontrados são menores que a oscilação da própria grade**.

A busca ampla revelou algo mais útil que o ganho: **no C1 o ramo do máximo nunca dispara**. De
`limite_max = 40` em diante o F2 não muda mais (0.3623 com 40, 60 ou `inf`), porque o C1 não tem
nenhum dia com máximo acima disso. O `30` escolhido antes estava na borda da grade e significava,
na prática, "ramo quase desligado".

In [ ]:
# Quanto se perde mexendo em cada limiar, com os outros dois fixos no ótimo
for _c in MAP_MEDICOES_BASE:
    print("=" * 78)
    sensibilidade_limiares(resultado_otimizacao, _c)
    print()

In [ ]:
# Qual ramo da regra carrega o resultado em cada reator
df_ablacao = ablacao_ramos_regra(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA,
    limiares_da_otimizacao(resultado_otimizacao), margem=margem_cross_reator
)

### Resposta 2: a margem é o que sustenta a regra — o ramo do máximo é acessório

A ablação separa a contribuição de cada ramo:

| Reator | regra completa | só o ramo do máximo | só mediana + margem | só mediana (sem margem) |
|---|---|---|---|---|
| C1 | **0.362** | 0.000 (nunca dispara) | **0.362** | 0.000 (0 VP, 60 FP) |
| C2 | **0.243** | 0.067 | 0.208 | 0.152 |
| C3 | **0.424** | 0.244 | 0.254 | 0.227 |

Duas leituras:

- **no C1 a regra inteira é o ramo mediana+margem** — e sem a margem ela vira ruído puro: zero
  detecções com 60 alarmes falsos. É a demonstração mais direta de que a comparação entre
  reatores, e não o nível de ferro, é o que carrega o sinal;
- **no C3 os dois ramos são complementares**: cada um sozinho pega 2 ou 3 falhas, juntos pegam 5.

Correção de uma afirmação anterior deste notebook: eu havia justificado o ramo do máximo dizendo
que a mediana diária não pegaria o evento de 23/11/2013 no C3 (264 ppm em uma única amostra —
mediana do dia de apenas 2.95 ppm, margem 0.00). **Isso está errado**: o alarme tem janela de 15
dias, e outro dia dessa janela dispara o ramo da mediana. Na busca global, desligar o ramo do
máximo (`inf`) dá exatamente o mesmo VP/FP do melhor resultado — ele é redundante quando a
mediana está em 2.75.

### Resposta 3: calibrar não compensa — e essa é a descoberta importante

Até aqui tudo foi medido **dentro da amostra**. A validação abaixo calibra usando só o passado de
cada corte e mede no bloco seguinte. As regras de referência (limiar fixo, sem calibração) são
medidas **nas mesmas fatias de teste**.

In [ ]:
df_wf, resumo_wf, consolidado_wf = validar_calibracao_walkforward(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, margem=margem_cross_reator, n_blocos=3
)

In [ ]:
# Quantos graus de liberdade a base sustenta? Mesmos cortes de tempo para todas.
df_estrategias, resumo_estrategias = comparar_estrategias_calibracao(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA, margem=margem_cross_reator
)

### Conclusão: o limiar fixo único vence, e a calibração por reator deve ser abandonada

**Fora da amostra, consolidando os três reatores:**

| Regra | VP | FP | precisão | recall | F2 fora da amostra |
|---|---|---|---|---|---|
| **composta fixa (20 / 3.0 / 0.6)** | 9 | 52 | 0.148 | 0.265 | **0.229** |
| simplificada (mediana>2.75 E margem>0.6) | 8 | 56 | 0.125 | 0.235 | 0.200 |
| **calibrada por reator no passado** | 6 | 46 | 0.115 | 0.176 | **0.159** |
| margem fixa (> 0.8) | 5 | 38 | 0.116 | 0.147 | 0.140 |
| regra vigente (`max > 5`) | 7 | 138 | 0.048 | 0.206 | 0.124 |

E o desempenho cai **monotonicamente com o número de parâmetros estimados**:

| Estratégia | parâmetros | F2 fora da amostra |
|---|---|---|
| fixa | **0** | **0.273** |
| só a margem, por reator | 3 | 0.237 |
| os 3 limiares, global | 3 | 0.232 |
| os 3 limiares, por reator | 9 | 0.213 |
| só a mediana, por reator | 3 | 0.182 |

O otimismo da calibração é enorme: ela promete F2 médio de **0.51** no treino e entrega **0.16**
no teste (C2: 0.46 → 0.06; C3: 0.67 → 0.21). Com 9 a 16 falhas por reator, escolher três números
por equipamento é decorar o histórico.

**Ressalva honesta:** a linha "fixa" leva uma vantagem embutida — os limiares 20/3.0/0.6 foram
escolhidos olhando a série inteira, então ela não é totalmente cega ao teste. A leitura correta é
"**não há evidência de que calibrar ajude, e há evidência de que atrapalha**", e não "0.229 é o
desempenho garantido do limiar fixo".

**Consequência para a entrega — isto reverte a recomendação da seção anterior:**

1. **um único conjunto de limiares para os três reatores** (`max > 20` **OU** (`mediana > 3.0`
   **E** `margem > 0.6`)), não uma tabela de três linhas. Menos parâmetros, mais robusto, e mais
   simples de implantar;
2. **o relatório continua por reator** — a assimetria de desempenho entre C1, C2 e C3 é real e
   precisa ser declarada; o que não se sustenta é *calibrar* separado;
3. **mexer nos limiares só com dados novos.** A grade ampla já foi varrida: não há ganho
   escondido. O que destrava a próxima melhoria é mais evento rotulado e a classificação do modo
   de falha, não um número melhor de ppm.

## Quadro consolidado por reator

Uma linha por equipamento, com tudo que a equipe precisa ver junto. **Não há linha de total de
propósito:** a média entre reatores esconde exatamente o fato que decide a implantação.

In [ ]:
df_resumo_reator = resumo_por_reator(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA,
    df_janelas=df_janelas, margem=margem_cross_reator,
    limiares_por_reator=LIMIARES_POR_REATOR, df_lead=df_lead, contexto=CONTEXTO
)
# df_resumo_reator.to_csv(DIR_OUTPUT + "resumo_por_reator.csv", sep=";", index=False)
df_resumo_reator

### Leitura reator a reator

**Crystallizer #3 — onde o método funciona, com um alcance agora medido com honestidade.**
9 falhas, 5 detectadas (recall 0.556) com 16 alarmes falsos, F2 **0.439**. As 5 detectadas são
as falhas "clássicas" de revestimento/casco — incluindo as duas paradas que a planilha atribui
ao ferro (264 ppm em 2013 e 999 ppm em 2018, ambas com furo confirmado). As 4 perdidas incluem
os **3 vazamentos em plug de reparo resgatados na auditoria (2023-2024), que não mostram ferro**
— vazamento pequeno em reparo de tântalo expõe pouco aço ao licor. O lead mediano é de 2 dias:
no C3 o ferro **confirma**, não antecipa.

**Crystallizer #1 — sinal na margem, com antecedência.** 12 falhas, 5 detectadas, F2 0.338.
Responde à margem cross-reator (lift 11.2 acima de 1.2), não ao nível. É o reator com **melhor
lead time: mediana de 15 dias** nas falhas alarmadas — onde existe janela real de decisão.

**Crystallizer #2 — o ferro não prevê, e provavelmente não é problema de método.** 16 falhas
(a auditoria acrescentou as trocas de revestimento constatadas em 2012 e 2016), 4 detectadas,
F2 0.222 mesmo com limiar próprio. Os apontamentos são dominados por **agitador e bocal**
(*"quebra dos parafusos da pá superior"*, *"Dano na Hélice"*, *"vazamento pela região do
selo/mesa"*) — modos em que ferro no licor não tem por que subir. O C2 tem **0 menções a ferro**
em 16 paradas, contra 2 no C3 em 9.

**Consequência:** parte do que estamos exigindo que o ferro preveja não é previsível por ferro,
**por construção** — e o resgate de eventos deixou isso visível também no C3. A correção não é
algorítmica: é a **classificação do modo de falha** de cada evento pela planta (coluna a
acrescentar na ficha de validação). Com ela, o recall passa a ser medido sobre o denominador
certo: as falhas que liberam ferro.

# Modo de falha: o que o ferro tem como enxergar

Todo resultado até aqui cobra do detector **as 36 falhas**. Mas o ferro só sobe no licor quando o
revestimento vitrificado rompe e expõe aço carbono à solução. Falha de agitador sem exposição de
aço, vazamento externo por junta ou selo, e vazamento em plug de reparo (área minúscula) **não têm
por que mover a medição** — exigir que o detector as antecipe é medir contra um denominador
impossível.

Esta seção testa o efeito de remover esses eventos do denominador. São dois cortes:

- **objetivo**, por `TipoFalha` — campo derivado de marcadores estruturados da planilha
  (`Emergencia` + a lógica de parada de planta), sem interpretação de texto. A hipótese é que uma
  falha *constatada em inspeção programada* tem menos relação com o ferro: o dano foi encontrado na
  abertura, pode ser antigo e pequeno, e não há razão para coincidir com uma excursão nos 15 dias
  anteriores;
- **proxy textual**, por `ModoFalha` — `classificar_modo_falha` lê OCORRIMENTO e OBSERVAÇÕES (o que
  aconteceu) e **não** SERVIÇOS EXECUTADOS (o que foi feito). A distinção importa: instalar um plug
  é o *reparo* de um furo, não o modo de falha; ignorá-la classificava 21 das 36 falhas como
  "plug", inclusive as duas paradas que a planilha atribui ao ferro.

**Isto é um proxy, não um laudo.** A classificação definitiva tem de vir da planta — é o pedido
nº 1 da entrega, e a coluna `ModoFalha` já viaja na ficha de validação para eles corrigirem. Duas
correções documentadas já estão aplicadas (`CORRECOES_MODO_FALHA`), nos dois eventos em que o
OCORRIMENTO descreve o *gatilho* da parada (a análise de ferro) e não o dano.

In [ ]:
# `df_inspecoes` já sai classificado da seção de eventos
_sel = df_inspecoes[df_inspecoes["Selecionado"]]

print("Falhas por modo (proxy textual):")
print(pd.crosstab(_sel["ModoFalha"], _sel["Crystallizer"], margins=True).to_string())
print(f"\nFerro fisicamente plausível (aço exposto ao licor): "
      f"{int(_sel['FerroPlausivel'].sum())} de {len(_sel)}")
print(f"Correções documentadas aplicadas: {int(_sel['ModoCorrigido'].sum())}")

print("\nClassificação evento a evento — é esta lista que a planta deve revisar:")
_sel.assign(texto=(_sel["Ocorrimento"] + " " + _sel["Observacoes"])
            .str.replace(r"\s+", " ", regex=True).str.slice(0, 60))[
    ["Crystallizer", "TS_Ajustado", "TipoFalha", "ModoFalha", "FerroPlausivel",
     "ModoCorrigido", "texto"]].sort_values(["Crystallizer", "TS_Ajustado"])

In [ ]:
REGRA_ENTREGA = (20, 3.0, 0.6)

print("=" * 92)
print("A. CORTE OBJETIVO — como a falha foi descoberta (campo estruturado)".center(92))
print("=" * 92)
df_modo_tipo = avaliar_regra_por_subconjunto(
    MAP_MEDICOES_BASE, df_inspecoes, REGRA_ENTREGA, "TipoFalha", margem=margem_cross_reator)

print()
print("=" * 92)
print("B. CORTE POR MODO DE FALHA (proxy textual)".center(92))
print("=" * 92)
df_modo_falha = avaliar_regra_por_subconjunto(
    MAP_MEDICOES_BASE, df_inspecoes, REGRA_ENTREGA, "ModoFalha", margem=margem_cross_reator)

print()
print("=" * 92)
print("C. O FERRO É FISICAMENTE PLAUSÍVEL NESSE MODO?".center(92))
print("=" * 92)
df_modo_ferro = avaliar_regra_por_subconjunto(
    MAP_MEDICOES_BASE, df_inspecoes, REGRA_ENTREGA, "FerroPlausivel", margem=margem_cross_reator)

### Resultado: o recall sobe, o F2 não — e isso corrige uma expectativa deste estudo

| Denominador | falhas | VP | recall | F2 |
|---|---|---|---|---|
| **todas** | 36 | 12 | 0.333 | **0.267** |
| só emergências | 18 | 8 | **0.444** | 0.268 |
| só constatadas em inspeção programada | 16 | 4 | 0.250 | 0.146 |
| só revestimento/casco (proxy) | 17 | 8 | **0.471** | 0.276 |
| só modos com ferro plausível | 24 | 9 | 0.375 | 0.259 |

**O recall sobe de 0.333 para 0.44–0.47 quando o denominador fica restrito ao que o ferro tem como
enxergar.** É uma diferença real e ela confirma a hipótese: as falhas constatadas em inspeção
programada são detectadas em 25% dos casos, contra 44% das emergências.

**Mas o F2 praticamente não se move (0.267 → 0.268 / 0.276).** O motivo é aritmético e importante:
**restringir o denominador não remove nenhum alarme falso.** O detector dispara o mesmo tanto; o
que muda é apenas quantas das falhas restantes ele acerta. Ganha-se em recall e perde-se em
precisão na mesma medida.

Isso **corrige uma expectativa registrada antes neste notebook** — eu havia escrito que "filtrando
para falhas de revestimento, o denominador do recall cai e o desempenho medido sobe". Sobe o
recall; o desempenho global, não.

**O que isso significa para a entrega:**

1. **O escopo declarado melhora a promessa, não o produto.** Dizer "esta regra cobre falha de
   revestimento, e nesse escopo acerta ~47%" é mais honesto e mais alto que "acerta 33% das
   falhas" — mas a carga de alarme falso que a operação sente é exatamente a mesma;
2. **Nem toda a distância até 100% é limitação física do sinal.** Mesmo no subconjunto mais
   favorável o detector perde mais da metade das falhas. A classificação de modo explica parte do
   gap, não o gap inteiro;
3. **A classificação de modo continua valendo o pedido à planta** — mas pelo motivo certo: ela
   define o *escopo contratual* do alarme e permite excluir do acompanhamento os modos que o ferro
   comprovadamente não vê, e não porque vá melhorar o número de F2.

---

# Resumo executivo — leitura para a equipe Bayer

*Documento de fechamento desta etapa. Todos os números abaixo são reproduzíveis pelas células
deste notebook e foram medidos sobre os mesmos dados: as três séries de ferro do laboratório
(2011-2026) e a planilha de inspeção dos vitrificados.*

## 1. A pergunta

O ferro medido no licor consegue **antecipar a falha do vitrificado** dos reatores de PIA, com
menos alarme falso do que a regra em uso (ferro acima de 5 ppm)?

O estudo anterior da equipe Bayer respondeu que não: dos 10 eventos analisados, **1** tinha ferro
acima de 5 ppm antes; e entre 2017 e 2025 houve **256 ultrapassagens, 247 sem falha** — cerca de
**3,5% de precisão**. Este trabalho parte daí, refaz a base de eventos e mede tudo contra taxa
base.

## 2. O que mudou na montagem do problema

Quatro correções que alteram qualquer conclusão posterior:

1. **A base de eventos foi refeita a partir da planilha de inspeção**, não do deck. Dos 100
   apontamentos, 60 caem no período da série; apontamentos do mesmo reator a menos de 7 dias
   foram fundidos; paradas de planta (lacuna simultânea nos três reatores) não entram como falha
   de equipamento — **exceto quando há dano físico constatado na abertura** (furo, infiltração),
   caso em que o evento é mantido com a âncora na última medição antes da parada. Resultado:
   **37 falhas** (C1 12, C2 16, C3 9).
2. **Os eventos foram reancorados para o fim da amostragem**, porque quando o reator para o
   laboratório também para: 39 dos 60 apontamentos caíam dentro de uma lacuna. A reancoragem
   inclusive **reconcilia divergências de data com o deck**: no evento do bocal do fundo do C1,
   o deck diz 19/01/2024 e a planilha 02/02/2024 — a amostragem para em **20/01**, confirmando o
   deck.
3. **A auditoria da planilha (11/08/2026) recuperou 6 falhas** que os marcadores de texto
   perdiam: vidro rompido até o metal (C1 2015), trocas de reator registradas em voz ativa
   (C2 2012 — reator com 308 dias de operação — e C2 2016), furo e infiltração constatados em
   parada de planta (C3 2023 e 2024) e o vazamento no plug que o próprio deck registra
   (C3 out/2024). As falhas **pós-2019 dobraram de 3 para 6**.
4. **Só o valor fisicamente impossível é descartado** (uma amostra de 20000 ppm em toda a base).
   As leituras altas isoladas ficam: são ultrapassagens que não terminaram em falha, ou seja, o
   falso positivo que qualquer detector precisa enfrentar. O corte antigo de 100 ppm apagava
   justamente as duas leituras que a planilha registra como **causa** de parada de emergência
   (264 ppm em 23/11/2013 e 999 ppm em 02/08/2018, ambas no C3, ambas com furo confirmado).

## 3. Principais achados

**1) A regra de 5 ppm não funciona — mas não porque o ferro seja inútil.** Na base auditada ela
dispara em 21,6% das janelas que antecedem falha e em 23,2% de uma janela qualquer: **lift 0,93**,
ou seja, dispara *menos* antes de falha do que no dia a dia. O limiar está no lugar errado, não a
variável.

**2) O que informa não é o nível de ferro — é a comparação entre os reatores.** A **margem**
(mediana diária do reator menos a mediana dos três) tem lift **3,8x a 5,8x**, contra 3,1x do melhor
critério de nível absoluto. No C1, a regra sem a margem produz **zero detecções com 60 alarmes
falsos**; com ela, 5 detecções em 11 falhas.

**3) A série caiu ao longo dos anos e isso contamina toda análise ingênua.** Tendência de queda
confirmada (tau de Kendall −0,29; mediana diária de 2,45-2,50 ppm em 2011-2015 para 1,80-1,90 em
2021-2026). Como **83% das falhas ocorreram até 2018**, ferro e falha caem juntos sem que um cause
o outro. Corrigindo por época, a "melhor feature" (mediana da janela) cai de AUC 0,627 para
**0,507** — era um relógio.

**4) Uma regra composta supera a regra vigente com folga**, e os limiares em uso já são os ótimos:
uma busca em 1100 combinações por reator não encontrou ganho maior que a oscilação da grade.

**5) Calibrar por reator não se sustenta.** Parece melhor dentro da amostra (F2 0,323 contra
0,267), mas na validação walk-forward entrega **0,159 contra 0,229** do limiar fixo — e o
desempenho piora monotonicamente com o número de parâmetros estimados. A entrega é **uma regra
única**, com relatório por reator.

**6) Modelos de machine learning não superaram a regra.** Fora da amostra e no mesmo período,
F2 0,146 do melhor modelo contra 0,203 da regra composta e 0,085 da vigente. Clusterização e
IsolationForest foram testados e descartados com a medição registrada.

**7) O desempenho é muito diferente entre reatores — e entre modos de falha.** As falhas que o
ferro vê são as de revestimento/casco; vazamentos em plug de reparo e falhas de agitador/bocal não
liberam ferro mensurável. É o achado mais acionável do estudo.

In [ ]:
# Tabela de fechamento. Esta célula é AUTO-SUFICIENTE de propósito: ela depende apenas
# dos objetos da seção de eventos e recalcula o resto se a seção "Resultados por reator"
# não tiver sido executada neste kernel — é a célula que vai para o relatório, e ela não
# pode quebrar por ordem de execução.
REGRA_ENTREGA = (20, 3.0, 0.6)   # max diário / mediana diária / margem cross-reator

if "margem_cross_reator" not in globals():
    _, _, margem_cross_reator = calcular_margem_cross_reator(MAP_MEDICOES_BASE, verbose=False)

print("=" * 78)
print("DESEMPENHO GLOBAL — 36 falhas, alarmes agrupados em 15 dias".center(78))
print("=" * 78)
df_fechamento = comparar_detectores(MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA,
                                    margem=margem_cross_reator)
print(df_fechamento.head(6).to_string(index=False))

print()
print("=" * 78)
print("DESEMPENHO POR REATOR — regra única de entrega".center(78))
print("=" * 78)
df_fechamento_reator = comparar_limiares_por_reator(
    MAP_MEDICOES_BASE, MAP_EVENTOS_FALHA,
    conjuntos={"regra de entrega": {c: REGRA_ENTREGA for c in MAP_MEDICOES_BASE}},
    margem=margem_cross_reator, verbose=False,
)
print(df_fechamento_reator.to_string(index=False))

# o número que a operação sente: alarmes falsos por reator por ano
_anos = np.mean([(d["TIMESTAMP"].max() - d["TIMESTAMP"].min()).days / 365.25
                 for d in MAP_MEDICOES_BASE.values()])
_fp_entrega = df_fechamento_reator["FP"].sum()
_fp_vigente = int(df_fechamento.loc[df_fechamento["regra"].str.contains("vigente"), "FP"].iloc[0])
print(f"\nCarga de alarme falso ao longo de {_anos:.0f} anos e {len(MAP_MEDICOES_BASE)} reatores:")
print(f"  regra de entrega : {_fp_entrega} alarmes falsos "
      f"= {_fp_entrega / (_anos * len(MAP_MEDICOES_BASE)):.1f} por reator por ano")
print(f"  regra vigente    : {_fp_vigente} alarmes falsos "
      f"= {_fp_vigente / (_anos * len(MAP_MEDICOES_BASE)):.1f} por reator por ano")

### O que a tabela por reator diz

| Reator | Situação | Recomendação |
|---|---|---|
| **C3** | 5 de 9 falhas detectadas (recall 0.556), F2 **0.439**, 16 alarmes falsos. As 5 detectadas são as falhas clássicas de revestimento — as 4 perdidas incluem os 3 vazamentos em plug (2023-2024), modo que não libera ferro. Lead mediano 2 dias. | **Implantar**, declarando o alcance: confirma falha de revestimento com muito menos alarme falso; não cobre vazamento em plug. |
| **C1** | 5 de 12 falhas, F2 0.338. Responde à **margem**, não ao nível. **Lead mediano de 15 dias** nas falhas alarmadas. | **Implantar em modo de acompanhamento.** É onde existe janela real de decisão. |
| **C2** | 4 de 16 falhas, F2 0.222 mesmo com limiar próprio. | **Não implantar ainda** — classificar o modo de falha antes. |

**A hipótese mais provável para o C2 não é estatística.** Os apontamentos são dominados por
falhas de **agitador e bocal** (*"quebra dos parafusos da pá superior do Hidro#1"*, *"Dano na
Hélice"*, *"vazamento pela região do selo/mesa"*) — e ferro no licor não tem por que subir
nesses modos. O C2 tem **0 menções a ferro** em 16 paradas; o C3 tem 2 em 9.

## 4. Limitações que precisam ser declaradas junto com o resultado

Estas ressalvas fazem parte da entrega. Nenhuma delas invalida os achados, mas todas mudam o que
se pode prometer:

1. **Os limiares foram escolhidos olhando as mesmas 37 falhas.** O bootstrap dá F2 = 0.256 com
   intervalo de 95% entre **0.156 e 0.363** — o piso fica acima da regra vigente (0.122), mas o
   intervalo é largo.
2. **As falhas recentes não são detectadas.** Depois de 01/01/2019 existem **6 falhas** (o
   resgate de eventos dobrou o número) e nenhuma regra acerta nenhuma — a vigente inclusive gera
   86 alarmes falsos no período. Os modos recentes (vazamento em plug de reparo, falha constatada
   em inspeção) **liberam pouco ferro**. Não é só falta de poder estatístico: é um limite físico
   do sinal, e delimita o escopo do produto.
3. **A antecipação é parcial.** 17 das 37 falhas recebem alarme em até 60 dias, lead mediano de
   3 dias, 8 falhas com uma semana ou mais. O produto honesto é **"antecipação parcial +
   confirmação com muito menos alarme falso"**, não "predição de falha".
4. **As datas e classificações dos eventos ainda precisam de confirmação.** A ficha de validação
   (60 apontamentos, com data original, data reancorada, deslocamento, tipo de falha e as duas
   correções documentadas) existe para ser revisada linha a linha pela planta.
5. **Não há variável de processo nem histórico de condenação.** Sem custo de alarme falso e
   custo de parada não planejada, o ponto de operação é escolhido por F2. A função de custo está
   montada: **basta a planta informar a razão de custo.**

## 5. O que pedimos à planta para a próxima etapa

Em ordem de impacto sobre o resultado:

1. **Classificação do modo de falha de cada evento** (revestimento/casco, plug de reparo, bocal,
   agitador, outros) — define o denominador correto do recall nos três reatores e explica o C2;
2. **Confirmação das datas** dos 60 apontamentos da ficha de validação — incluindo os dois casos
   em que a reancoragem contradiz ou corrige a planilha (bocal do fundo do C1 em 2024; plug do
   C3 em out/2024) e a substituição da BV de 2017 que não tem registro na planilha;
3. **Custo de um alarme falso** (condenação indevida) e **custo de uma parada não planejada** —
   define o ponto de operação;
4. **Variáveis de processo** do circuito (temperatura, pressão, concentração de soda, vazão) —
   único caminho para o aprendizado de máquina voltar a fazer sentido;
5. **Rótulos de falha adicionais pós-2019**, se existirem em outra fonte — o resgate dobrou de
   3 para 6, mas todas as 6 são de modos com pouco ferro.

## 6. O que está pronto para entrega

**Uma regra única para os três reatores** — não um modelo, e não uma tabela de limiares por
equipamento. Duas condições sobre estatísticas diárias do próprio laboratório, implementáveis em
qualquer linguagem da planta e auditáveis pela operação:

```
ALARME  se   máximo diário de ferro > 20 ppm
        ou  ( mediana diária > 3,0 ppm  E  margem sobre a mediana dos três reatores > 0,6 ppm )
```

Onde **margem = mediana diária do reator − mediana diária dos três reatores no mesmo dia** — a
única grandeza deste estudo que exige os três reatores medidos no mesmo dia, e a que mais carrega
informação (no C1, sem ela a regra vira ruído: zero detecções com 60 alarmes falsos).

**Por que um limiar só, e não um por reator.** A calibração por equipamento parecia melhor dentro
da amostra (F2 0.323 contra 0.267), mas na validação *walk-forward* — calibrar no passado, medir
no futuro — o resultado se inverte: **0.159 calibrado contra 0.229 fixo**. O desempenho cai
monotonicamente com o número de parâmetros estimados (0 params: 0.273; 3 params: 0.23; 9 params:
0.213). Com 9 a 16 falhas por reator, escolher três números por equipamento é decorar o histórico.
Uma busca em grade ampla (1100 combinações por reator) confirmou que **os limiares em uso já são
ótimos dentro do ruído** — os ganhos disponíveis (+0.005 a +0.021 de F2) são menores que a
oscilação da própria grade.

**O escopo, que vai junto com a regra:**

| Reator | Desempenho medido | Recomendação |
|---|---|---|
| C3 | recall 0,556 (5 de 9), F2 0,439 — as 5 são falhas de revestimento; as 4 perdidas incluem 3 vazamentos em plug, que não liberam ferro | **implantar** para falha de revestimento, declarando que não cobre vazamento em plug |
| C1 | recall 0,455 (5 de 11), F2 0,357, **lead mediano de 15 dias** | **implantar em acompanhamento** — é onde existe janela real de decisão |
| C2 | recall 0,250 (4 de 16), F2 0,222; paradas dominadas por agitador e bocal | **não implantar** antes da classificação do modo de falha |

**O que NÃO fazer:** mexer nos limiares com os dados atuais. A grade já foi varrida por inteiro e
não há ganho escondido; qualquer ajuste fino a partir daqui é ruído. O que destrava a próxima
melhoria é **mais evento rotulado** e a **classificação do modo de falha**, não um número melhor
de ppm.

Junto com a regra vão: a **ficha de validação de eventos** (com as falhas resgatadas, as âncoras
da planilha e as correções documentadas), a **tabela de sensibilidade a custo** e este notebook,
com cada decisão documentada na célula em que foi tomada.